In [1]:
!python --version

Python 3.12.13


In [2]:
!pip install transformers datasets torch pandas Faker scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 26.8 MB/s eta 0:00:00


In [3]:
# from google.colab import drive
# drive.mount('/content/drive')

In [4]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import torch
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [5]:
print(f"Current working directory: {os.getcwd()}")
print(f"Directories found: {[d for d in os.listdir('.') if os.path.isdir(d)]}")

Current working directory: /kaggle/working
Directories found: []


In [ ]:
CWD = './'
# CP_PATH = os.path.join(CWD,'stage1_toxic_ftall_cp','checkpoint-179205')
# TRAINED_PATH = os.path.join(CWD,'stage1_toxic_ft6')

DS_SIZE = 100000
SAFE_RATIO = 0.3
EVAL_RATIO = 0.1



In [7]:
# from datasets import load_dataset
# import pandas as pd

# def prepare_toxic_class():
#     # --- 1. Process VOZ-HSD (Vietnamese) ---
#     print("Loading VOZ-HSD...")
#     voz_ds = load_dataset("tarudesu/VOZ-HSD", split='train')
#     voz_df = voz_ds.to_pandas()

#     # SQL: select texts from train where labels = 1 order by probs desc limit 36000
#     voz_toxic = (
#         voz_df[voz_df['labels'] == 1]                 # Filter for toxic
#         .sort_values(by='probs', ascending=False)     # Order by probs desc
#         .head(NUM_TOXIC)                                  # Limit 24000
#     )

#     # Standardize to our Omni-Filter format
#     voz_final = voz_toxic[['texts']].rename(columns={'texts': 'text'}).assign(label=1)

#     # --- 2. Process Civil Comments (English) ---
#     print("Loading Civil Comments...")
#     # (Keeping your specific 0.5-0.75 and >0.75 bucket requirements)
#     def filter_civil(split_name, n_total):
#         ds = load_dataset("google/civil_comments", split=split_name)
#         df = ds.to_pandas()

#         mild = df[(df['toxicity'] > 0.5) & (df['toxicity'] <= 0.75)].head(n_total // 2)
#         severe = df[df['toxicity'] > 0.75].head(n_total // 2)

#         return pd.concat([mild, severe])[['text']].assign(label=1)

#     civil_train = filter_civil('train', int(NUM_TOXIC * 0.8))
#     civil_val = filter_civil('validation', int(NUM_TOXIC * 0.1))
#     civil_test = filter_civil('test', int(NUM_TOXIC * 0.1))

#     # --- 3. 8/1/1 Split for VOZ ---
#     voz_train = voz_final.sample(frac=0.8, random_state=42)
#     voz_remaining = voz_final.drop(voz_train.index)
#     voz_val = voz_remaining.sample(frac=0.5, random_state=42)
#     voz_test = voz_remaining.drop(voz_val.index)

#     # --- 4. Final Merge, Reorder & Shuffle ---
#     # Combine EN and VN for each split
#     train_df = pd.concat([civil_train, voz_train])
#     val_df = pd.concat([civil_val, voz_val])
#     test_df = pd.concat([civil_test, voz_test])

#     # Reorder columns to [label, text] and shuffle
#     train_df = train_df[['label', 'text']].sample(frac=1, random_state=42).reset_index(drop=True)
#     val_df = val_df[['label', 'text']].sample(frac=1, random_state=42).reset_index(drop=True)
#     test_df = test_df[['label', 'text']].sample(frac=1, random_state=42).reset_index(drop=True)

#     print("Columns swapped. Format is now: [label, text]")
#     print(f"Final Dataset Ready: {len(train_df)} train, {len(val_df)} val, {len(test_df)} test.")
#     return train_df, val_df, test_df

# toxic_train, toxic_val, toxic_test = prepare_toxic_class()

# toxic_train.head()

In [8]:
# from datasets import load_dataset
# import pandas as pd

# def prepare_toxic_class():
#     # --- 1. Process VOZ-HSD (Vietnamese) ---
#     print("Loading VOZ-HSD...")
#     voz_ds = load_dataset("tarudesu/VOZ-HSD", split='train')
#     voz_df = voz_ds.to_pandas()

#     # SQL: select texts from train where labels = 1 order by probs desc limit 36000
#     voz_toxic = (
#         voz_df[voz_df['labels'] == 1]                 # Filter for toxic
#         .sort_values(by='probs', ascending=False)     # Order by probs desc
#         .head(NUM_TOXIC)                                  # Limit 24000
#     )
#     # SQL: select texts from train where labels = 1 order by probs desc limit 36000
#     voz_safe = (
#         voz_df[voz_df['labels'] == 0]                 # Filter for toxic
#         .sort_values(by='probs', ascending=False)     # Order by probs desc
#         .head(NUM_SAFE)                                  # Limit 24000
#     )

#     # Standardize to our Omni-Filter format

#     voz_safe = voz_safe[['texts']].rename(columns={'texts':'text'}).assign(label=0)
#     voz_toxic = voz_toxic[['texts']].rename(columns={'texts':'text'}).assign(label=1)

#     def split_voz(voz_ds):
#         # --- 3. 8/1/1 Split for VOZ ---

#         voz_train = voz_ds.sample(frac=TRAIN_RATIO, random_state=42)
#         voz_remaining = voz_ds.drop(voz_train.index)
#         voz_val = voz_remaining.sample(frac=0.5, random_state=42)
#         voz_test = voz_remaining.drop(voz_val.index)

#         print(f'train : {len(voz_train)}')
#         print(f'val : {len(voz_val)}')
#         print(f'test : {len(voz_test)}')

#         return voz_train,voz_val,voz_test

#     print('VOZ_SAFE:')
#     voz_safe_train, voz_safe_val, voz_safe_test = split_voz(voz_safe)
#     print('VOZ_TOXIC:')
#     voz_toxic_train, voz_toxic_val, voz_toxic_test = split_voz(voz_toxic)

#     voz_final_train = pd.concat([voz_safe_train,voz_toxic_train])
#     voz_final_val = pd.concat([voz_safe_val,voz_toxic_val])
#     voz_final_test = pd.concat([voz_safe_test,voz_toxic_test])


#     # --- 2. Process Civil Comments (English) ---
#     print("Loading Civil Comments...")
#     # (Keeping your specific 0.5-0.75 and >0.75 bucket requirements)
#     def filter_civil(split_name, n_toxic, n_safe):
#         ds = load_dataset("google/civil_comments", split=split_name)
#         df = ds.to_pandas()

#         safe = df[df['toxicity'] == 0].head(n_safe)[['text']].assign(label=0)


#         mild = df[(df['toxicity'] > 0) & (df['toxicity'] < 0.4)].head(n_toxic // 3)
#         average = df[(df['toxicity'] >= 0.4) & (df['toxicity'] < 0.7)].head(n_toxic // 3)
#         severe = df[df['toxicity'] >= 0.7].head(n_toxic // 3)

#         toxic_concat = pd.concat([mild, average, severe])[['text']].assign(label=1)

#         print(f'civil_safe ({split_name}): {len(safe)}')
#         print(f'civil_mild ({split_name}): {len(mild)}')
#         print(f'civil_average ({split_name}): {len(average)}')
#         print(f'civil_severe ({split_name}): {len(severe)}')


#         return pd.concat([safe,toxic_concat])

#     civil_train = filter_civil('train', int(NUM_TOXIC * TRAIN_RATIO), int(NUM_SAFE * TRAIN_RATIO))
#     civil_val = filter_civil('validation', int(NUM_TOXIC * VAL_TEST_RATIO), int(NUM_SAFE* VAL_TEST_RATIO))
#     civil_test = filter_civil('test', int(NUM_TOXIC * VAL_TEST_RATIO), int(NUM_SAFE * VAL_TEST_RATIO))





#     # --- 4. Final Merge, Reorder & Shuffle ---
#     # Combine EN and VN for each split
#     train_df = pd.concat([civil_train, voz_final_train])
#     val_df = pd.concat([civil_val, voz_final_val])
#     test_df = pd.concat([civil_test, voz_final_test])

#     # Reorder columns to [label, text] and shuffle
#     train_df = train_df[['label', 'text']].sample(frac=1, random_state=42).reset_index(drop=True)
#     val_df = val_df[['label', 'text']].sample(frac=1, random_state=42).reset_index(drop=True)
#     test_df = test_df[['label', 'text']].sample(frac=1, random_state=42).reset_index(drop=True)

#     print("Columns swapped. Format is now: [label, text]")
#     print(f"Final Dataset Ready: {len(train_df)} train, {len(val_df)} val, {len(test_df)} test.")
#     return train_df, val_df, test_df

# train_df, val_df, test_df = prepare_toxic_class()
# train_df.head()
# # toxic_train, toxic_val, toxic_test = prepare_toxic_class()
# # toxic_train.head()

In [9]:
from datasets import load_dataset
import pandas as pd

def prepare_toxic_class():
  
    # --- 1. Process VOZ-HSD (Vietnamese) ---
    print("Loading VOZ-HSD...")
    voz_ds = load_dataset("tarudesu/VOZ-HSD", split='train')
    voz_df = voz_ds.to_pandas()

    safe_size = int(DS_SIZE * SAFE_RATIO)
    toxic_size = DS_SIZE - safe_size    
    
    voz_mild = (
        voz_df[(voz_df['labels'] == 1) & (voz_df['probs'] > 0.5) & (voz_df['probs'] < 0.7)]                 
        .head(toxic_size // 3)                                  
    )

    voz_moderate = (
        voz_df[(voz_df['labels'] == 1) & (voz_df['probs'] >= 0.7) & (voz_df['probs'] < 0.9)]                 
        .head(toxic_size // 3)                                  
    )

    voz_severe = (
        voz_df[voz_df['labels'] == 1]
        .sort_values(by='probs', ascending=False) 
        .head(toxic_size // 3)                                  
    )

  

    voz_safe = (
        voz_df[(voz_df['labels'] == 0) & (voz_df['probs'] > 0.995)]    # Filter for toxic
        .sort_values(by='probs', ascending=False)     # Order by probs desc
        .head(safe_size)                                  # Limit 
    )

    # Standardize to our Omni-Filter format

    voz_mild = voz_mild[['texts']].rename(columns={'texts':'text'}).assign(label=1)
    voz_moderate = voz_moderate[['texts']].rename(columns={'texts':'text'}).assign(label=1)
    voz_severe = voz_severe[['texts']].rename(columns={'texts':'text'}).assign(label=1)
    voz_safe = voz_safe[['texts']].rename(columns={'texts':'text'}).assign(label=0)


    def split_voz(voz_ds):
        # --- 3. 9/1/1 Split for VOZ ---

        voz_train = voz_ds.sample(frac=1-EVAL_RATIO, random_state=42)
        voz_remaining = voz_ds.drop(voz_train.index)
        voz_val = voz_remaining.sample(frac=0.5, random_state=42)
        voz_test = voz_remaining.drop(voz_val.index)
        
        print(f'Before split: {len(voz_ds)}')
        print(f'train : {len(voz_train)}')
        print(f'val : {len(voz_val)}')
        print(f'test : {len(voz_test)}')

        return voz_train,voz_val,voz_test

    print('VOZ_SAFE:')
    voz_safe_train, voz_safe_val, voz_safe_test = split_voz(voz_safe)
    print('VOZ_MILD:')
    voz_mild_train, voz_mild_val, voz_mild_test = split_voz(voz_mild)
    print('VOZ_MODERATE:')
    voz_moderate_train, voz_moderate_val, voz_moderate_test = split_voz(voz_moderate)
    print('VOZ_SEVERE:')
    voz_severe_train, voz_severe_val, voz_severe_test = split_voz(voz_severe)

    voz_final_train = pd.concat([voz_safe_train,voz_mild_train,voz_moderate_train,voz_severe_train])
    voz_final_val = pd.concat([voz_safe_val,voz_mild_val,voz_moderate_val,voz_severe_val])
    voz_final_test = pd.concat([voz_safe_test,voz_mild_test,voz_moderate_test,voz_severe_test])
    


    # --- 2. Process Civil Comments (English) ---
    print("Loading Civil Comments...")

    
    
    # (Keeping your specific 0.5-0.75 and >0.75 bucket requirements)
    def filter_civil(split_name, n_toxic, n_safe):
        ds = load_dataset("google/civil_comments", split=split_name)
        df = ds.to_pandas()

        safe = df[df['toxicity'] == 0].head(n_safe)[['text']].assign(label=0)


        mild = df[(df['toxicity'] > 0.1) & (df['toxicity'] < 0.4)].head(n_toxic // 3)
        average = df[(df['toxicity'] >= 0.4) & (df['toxicity'] < 0.7)].head(n_toxic // 3)
        severe = df[df['toxicity'] >= 0.7].head(n_toxic // 3)

        toxic_concat = pd.concat([mild, average, severe])[['text']].assign(label=1)

        print(f'CIVIL_{split_name.upper()}:')
        print(f'safe ({split_name}): {len(safe)}')
        print(f'mild ({split_name}): {len(mild)}')
        print(f'average ({split_name}): {len(average)}')
        print(f'severe ({split_name}): {len(severe)}')


        return pd.concat([safe,toxic_concat])

    civil_train = filter_civil('train', int(toxic_size * (1-EVAL_RATIO)), int(safe_size * (1-EVAL_RATIO)))
    civil_val = filter_civil('validation', int(toxic_size * EVAL_RATIO/2), int(safe_size* EVAL_RATIO/2))
    civil_test = filter_civil('test', int(toxic_size * EVAL_RATIO/2), int(safe_size * EVAL_RATIO/2))


    # --- 4. Final Merge, Reorder & Shuffle ---
    # Combine EN and VN for each split
    train_df = pd.concat([civil_train, voz_final_train])
    val_df = pd.concat([civil_val, voz_final_val])
    test_df = pd.concat([civil_test, voz_final_test])

    # Reorder columns to [label, text] and shuffle
    train_df = train_df[['label', 'text']].sample(frac=1, random_state=42).reset_index(drop=True)
    val_df = val_df[['label', 'text']].sample(frac=1, random_state=42).reset_index(drop=True)
    test_df = test_df[['label', 'text']].sample(frac=1, random_state=42).reset_index(drop=True)

    print("Columns swapped. Format is now: [label, text]")
    print(f"Final Dataset Ready: {len(train_df)} train, {len(val_df)} val, {len(test_df)} test.")
    return train_df, val_df, test_df

train_df, val_df, test_df = prepare_toxic_class()
train_df.head(10)
# toxic_train, toxic_val, toxic_test = prepare_toxic_class()
# toxic_train.head()

Loading VOZ-HSD...


README.md: 0.00B [00:00, ?B/s]

data.csv:   0%|          | 0.00/1.89G [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10747733 [00:00<?, ? examples/s]

VOZ_SAFE:
Before split: 30000
train : 27000
val : 1500
test : 1500
VOZ_MILD:
Before split: 23333
train : 21000
val : 1166
test : 1167
VOZ_MODERATE:
Before split: 23333
train : 21000
val : 1166
test : 1167
VOZ_SEVERE:
Before split: 23333
train : 21000
val : 1166
test : 1167
Loading Civil Comments...


...

Generating train split:   0%|          | 0/1804874 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/97320 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/97320 [00:00<?, ? examples/s]

CIVIL_TRAIN:
safe (train): 27000
mild (train): 21000
average (train): 21000
severe (train): 21000
CIVIL_VALIDATION:
safe (validation): 1500
mild (validation): 1166
average (validation): 1166
severe (validation): 1166
CIVIL_TEST:
safe (test): 1500
mild (test): 1166
average (test): 1166
severe (test): 1166
Columns swapped. Format is now: [label, text]
Final Dataset Ready: 180000 train, 9996 val, 9999 test.


,label,text
0,1,Luật công bằng tài chính do bọn nó nghĩ ra. Mà...
1,1,"sai lòi lol ra chứ ko sai, nó kêu tôi bán hàng..."
2,1,Is changing some clocks and waking up earlier/...
3,0,"Add churches and other tax-exempt ""non-profit""..."
4,1,Oh geeeze... make it end Alaska.\n\nIf you get...
5,1,"thôi thì mày lướt cmm đi, đạo đức giả"
6,1,"""Dân gian"" mà, xong lúc đéo nào cũng chê đất n..."
7,1,hate of democracy is one trait shared by stupi...
8,0,NO it wouldn't! It would be a relay of informa...
9,1,Over 50% of white women voted for Trump. What...


In [10]:
# train_df = pd.read_csv(os.path.join(CWD,'train.csv'),usecols=['label','text'])
# val_df = pd.read_csv(os.path.join(CWD,'val.csv'),usecols=['label','text'])
# test_df = pd.read_csv(os.path.join(CWD,'test.csv'),usecols=['label','text'])

print(f'Train: {len(train_df)}')
print(f'Val: {len(val_df)}')
print(f'Test: {len(test_df)}')

print(train_df)

train_df.to_csv(os.path.join(CWD,'train.csv'))
val_df.to_csv(os.path.join(CWD,'val.csv'))
test_df.to_csv(os.path.join(CWD,'test.csv'))

Train: 180000
Val: 9996
Test: 9999
        label                                               text
0           1  Luật công bằng tài chính do bọn nó nghĩ ra. Mà...
1           1  sai lòi lol ra chứ ko sai, nó kêu tôi bán hàng...
2           1  Is changing some clocks and waking up earlier/...
3           0  Add churches and other tax-exempt "non-profit"...
4           1  Oh geeeze... make it end Alaska.\n\nIf you get...
...       ...                                                ...
179995      1  Chết thằng PQ đi, toàn bọn bán cơm bụi HN ở nơ...
179996      0  Trên Beatvn, Theanh28 thì vẫn phát triển rực r...
179997      1                                @Thịt Chó Không Fen
179998      1  Sữa và đường,đéo có đéo uống.đéo có đá đéo uốn...
179999      1  Mua con blade 22 củ. Dm tầm tiền này đéo có co...

[180000 rows x 2 columns]


# Tokenizing data

In [11]:
MODEL_NAME = "distilbert-base-multilingual-cased"
# TRAINED_PATH = '/kaggle/input/models/kienanhuynhce191266/toxic-phase-1/transformers/default/1'
TRAINED_PATH = None
DEVICE = "cuda" if torch.cuda.is_available() else "cpu" 

In [12]:
# 3. Initialize Tokenizer (DistilBERT is the laptop king)
tokenizer = AutoTokenizer.from_pretrained(TRAINED_PATH if TRAINED_PATH else MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding='max_length', max_length=128)

# Convert to HuggingFace format
from datasets import Dataset
train_dataset = Dataset.from_pandas(train_df).map(tokenize_function, batched=True)
val_dataset = Dataset.from_pandas(val_df).map(tokenize_function, batched=True)
test_dataset = Dataset.from_pandas(test_df).map(tokenize_function, batched=True)

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/180000 [00:00<?, ? examples/s]

Map:   0%|          | 0/9996 [00:00<?, ? examples/s]

Map:   0%|          | 0/9999 [00:00<?, ? examples/s]

# Load model

In [13]:
# 4. Load Model
# When initializing the model:
# model = AutoModelForSequenceClassification.from_pretrained(CP_PATH).to(DEVICE)

id2label = {0: "SAFE", 1: "TOXIC"}
label2id = {"SAFE": 0, "TOXIC": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    TRAINED_PATH if TRAINED_PATH else MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    dropout=0.3,
    ignore_mismatched_sizes=True 
).to(DEVICE)




model.safetensors:   0%|          | 0.00/542M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# Class weights

In [14]:
import torch
from torch import nn
from transformers import Trainer

# Example weights based on your 168k total
# (Lower number = class is common, Higher number = give this class more attention)
def calc_weights(total_size, class_size, num_class = 2):
    return total_size/(num_class * class_size)

def get_class_weights(total_df, device, num_class = 2):
    
    ds_size = len(total_df)
    weight_values = []
    for i in range(num_class):
        cls_size = len(total_df[total_df['label'] == i])
        weight_values.append(calc_weights(ds_size,cls_size,num_class=num_class))
    
    return torch.tensor(weight_values).to(device)
    

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        # Call the original Trainer init
        super().__init__(*args, **kwargs)
        
        # Store your custom argument
        self.class_weights = class_weights
         
    def compute_loss(self, model, inputs, return_outputs=False,**kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        # Apply the weights to the CrossEntropyLoss
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# Metrics computation

In [15]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import torch.nn.functional as F

# Define the Metrics Function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    # Calculate metrics
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    _, _, f1_macro, _ = precision_recall_fscore_support(labels, predictions, average='macro')
    _, _, f1_weighted, _ = precision_recall_fscore_support(labels, predictions, average='weighted')

    acc = accuracy_score(labels, predictions)

    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
    }

# # Define the Metrics Function
# def compute_metrics(eval_pred):
#     logits, labels = eval_pred
#     predictions = np.argmax(logits, axis=-1)

#     # Calculate metrics
#     precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average=None)
#     precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(labels, predictions, average='macro')
#     precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(labels, predictions, average='weighted')

#     acc = accuracy_score(labels, predictions)

#     return {
#         'accuracy': acc,
#         'precision_class_0': precision[0],
#         'precision_class_1': precision[1],
#         'precision_class_2': precision[2],
#         'recall_class_0': recall[0],
#         'recall_class_1': recall[1],
#         'recall_class_2': recall[2],
#         'f1_class_0': f1[0],
#         'f1_class_1': f1[1],
#         'f1_class_2': f1[2],
#         'precision_macro': precision_macro,
#         'recall_macro': recall_macro,
#         'f1_macro': f1_macro,
#         'precision_weighted': precision_weighted,
#         'recall_weighted': recall_weighted,
#         'f1_weighted': f1_weighted,
#     }


# Early Stopping

In [16]:
callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]

# Print helper

In [17]:
def print_model_info(model):
    print(model)
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model: {total_params:,} total, {trainable_params:,} trainable")

In [18]:
# 5. Training Arguments (Aggressively optimized for your laptop)
training_args = TrainingArguments(
    output_dir='./toxic_cp_2',
    num_train_epochs=10, # 1 epoch is plenty for a base classifier
    per_device_train_batch_size=16, 
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    
    weight_decay=0.01,
    max_grad_norm=1.0,
    
    lr_scheduler_type="cosine",
    warmup_steps=0.2,
    
    logging_strategy = 'epoch',
    eval_strategy="epoch",
    save_strategy="epoch",
    # logging_strategy = 'steps',
    # logging_steps=50,
    # eval_strategy="steps",
    # eval_steps=50,
    # save_strategy="steps",
    # save_steps=50,
    
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1_macro",
    greater_is_better = True,
    
    fp16=torch.cuda.is_available(), # Use mixed precision if you have a GPU
    dataloader_num_workers=2,
    remove_unused_columns=True, 
    report_to="none",
    save_total_limit=3,
    dataloader_drop_last=False,
)


# Get class weights
total_df = pd.concat([train_df,val_df,test_df])
class_weights = get_class_weights(total_df, DEVICE, num_class=2)
print(f'Class weights: {class_weights}')

# 6. Initialize Trainer
trainer = WeightedTrainer(
    class_weights = class_weights,
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics, # Add this!
    callbacks = callbacks,
)

# 7. Start Training
print("Training Phase 1...")
trainer.train()

Class weights: tensor([1.6666, 0.7143], device='cuda:0')
Training Phase 1...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,F1 Macro,F1 Weighted
1,0.894356,0.301543,0.876851,0.923461,0.898513,0.910816,0.855958,0.877888
2,0.506976,0.238100,0.897559,0.949023,0.902087,0.924960,0.881799,0.899053
3,0.425183,0.242535,0.909064,0.940257,0.929102,0.934647,0.892607,0.909413
4,0.381979,0.233398,0.907863,0.949933,0.916667,0.933004,0.892763,0.908849
5,0.350011,0.248565,0.908663,0.942271,0.926244,0.934189,0.892491,0.909160
6,0.325109,0.276713,0.907863,0.938755,0.928959,0.933831,0.891088,0.908175
7,0.304391,0.303483,0.907263,0.933686,0.933819,0.933753,0.889613,0.907258


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=19691, training_loss=0.45542937393967264, metrics={'train_runtime': 11316.8023, 'train_samples_per_second': 159.056, 'train_steps_per_second': 2.486, 'total_flos': 4.172723057664e+16, 'train_loss': 0.45542937393967264, 'epoch': 7.0})

# Eval

In [19]:
def do_eval(trainer,eval_set):
    eval_res = trainer.evaluate(eval_dataset=eval_set)
    print(pd.Series(eval_res).head(100))

do_eval(trainer,val_dataset)



/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


eval_loss                    0.233418
eval_accuracy                0.907963
eval_precision               0.950074
eval_recall                  0.916667
eval_f1                      0.933071
eval_f1_macro                0.892889
eval_f1_weighted             0.908952
eval_runtime                24.512200
eval_samples_per_second    407.797000
eval_steps_per_second        6.405000
epoch                        7.000000
dtype: float64


# Test

In [20]:
# 8. Evaluate on the final Test Set
print("\n--- Final Evaluation on Test Dataset ---")

def do_test(trainer,ds,export_csv = False, export_filename = 'errors.csv'):
    # Run prediction on your test set
    preds = trainer.predict(ds)
    pred_labels = np.argmax(preds.predictions, axis=-1)
    true_labels = preds.label_ids
    metrics = preds.metrics
    
    # This is the "Truth Table"
    print(classification_report(true_labels, pred_labels, target_names=["SAFE", "TOXIC"]))
    # print(classification_report(true_labels, pred_labels, target_names=["SAFE", "SCAM", "TOXIC"]))
    df = pd.Series(metrics)
    print(df.head(100))

    if export_csv:        
        errors = []
        # Inspect misclassified examples
        for i, (true, pred) in enumerate(zip(true_labels, pred_labels)):
            if true != pred:
                errors.append({'idx':i, 'text': ds['text'][i], 'true': true, 'pred': pred})
                
        pd.DataFrame(errors).to_csv(export_filename)

do_test(trainer,test_dataset, export_csv = True, export_filename = 'errors_toxic.csv')



--- Final Evaluation on Test Dataset ---


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


              precision    recall  f1-score   support

        SAFE       0.83      0.88      0.86      3000
       TOXIC       0.95      0.92      0.94      6999

    accuracy                           0.91      9999
   macro avg       0.89      0.90      0.90      9999
weighted avg       0.91      0.91      0.91      9999

test_loss                    0.236884
test_accuracy                0.910591
test_precision               0.947646
test_recall                  0.923275
test_f1                      0.935302
test_f1_macro                0.895321
test_f1_weighted             0.911311
test_runtime                24.996600
test_samples_per_second    400.014000
test_steps_per_second        6.281000
dtype: float64


In [21]:
# Save the model and tokenizer to a local folder
model.save_pretrained("./toxic_phase_2")
tokenizer.save_pretrained("./toxic_phase_2")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./toxic_phase_2/tokenizer_config.json', './toxic_phase_2/tokenizer.json')

In [22]:
import shutil
shutil.make_archive('toxic_phase_2','zip','./toxic_phase_2')

'/kaggle/working/toxic_phase_2.zip'

# Fine-tune 2 layers

In [23]:
# # model = AutoModelForSequenceClassification.from_pretrained('/kaggle/working/stage1_freezed')
# # tokenizer = AutoTokenizer.from_pretrained('/kaggle/working/stage1_freezed')
# for p in model.distilbert.transformer.layer[-2:].parameters():
#     p.requires_grad = True

# print_model_info(model)

In [24]:
# # 5. Training Arguments (Aggressively optimized for your laptop)
# training_args = TrainingArguments(
#     output_dir="/content/drive/MyDrive/Colab Notebooks/stage1_toxic_ft2_cp",
#     num_train_epochs=7, # 1 epoch is plenty for a base classifier
#     per_device_train_batch_size=8,
#     per_device_eval_batch_size=32,
#     gradient_accumulation_steps=2,
#     learning_rate=5e-6,

#     weight_decay=0.01,
#     max_grad_norm=1.0,

#     lr_scheduler_type="cosine",
#     warmup_steps=0.1,

#     logging_strategy= 'epoch',
#     # logging_steps=1000,
#     eval_strategy="epoch",
#     # eval_steps=500,
#     save_strategy="epoch",
#     # save_steps=500,

#     load_best_model_at_end=True,
#     metric_for_best_model="eval_loss",
#     greater_is_better = False,

#     fp16=torch.cuda.is_available(), # Use mixed precision if you have a GPU
#     dataloader_num_workers=2,
#     # remove_unused_columns=True,
#     report_to="none",
#     save_total_limit=1,
#     dataloader_drop_last=False,
# )

# # 6. Initialize Trainer
# trainer = WeightedTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_dataset,
#     eval_dataset=val_dataset,
#     compute_metrics=compute_metrics, # Add this!
#     callbacks = callbacks
# )

# trainer.train()


In [25]:
# # 8. Evaluate on the final Test Set
# print("\n--- Final Evaluation on Test Dataset ---")


# # Run prediction on your test set
# preds = trainer.predict(test_dataset)
# pred_labels = np.argmax(preds.predictions, axis=-1)
# true_labels = preds.label_ids
# metrics = preds.metrics

# # This is the "Truth Table"
# print(classification_report(true_labels, pred_labels, target_names=["SAFE", "TOXIC"]))
# # print(classification_report(true_labels, pred_labels, target_names=["SAFE", "SCAM", "TOXIC"]))
# print(metrics)

In [26]:
# # Save the model and tokenizer to a local folder
# model.save_pretrained("./stage1_toxic_ft2")
# tokenizer.save_pretrained("./stage1_toxic_ft2")

# Finetune all transformer layers

In [27]:
# for p in model.distilbert.transformer.layer[0:].parameters():
#     p.requires_grad = True

# print_model_info(model)



In [28]:
# # 5. Training Arguments (Aggressively optimized for your laptop)
# training_args = TrainingArguments(
#     output_dir="/content/drive/MyDrive/Colab Notebooks/stage1_toxic_ft6_cp",
#     resume_from_checkpoint=os.path.join(CWD,'stage1_toxic_ft6_cp','checkpoint-35841'),
#     num_train_epochs=7, # 1 epoch is plenty for a base classifier
#     per_device_train_batch_size=8,
#     per_device_eval_batch_size=32,
#     gradient_accumulation_steps=2,
#     learning_rate=1e-6,

#     weight_decay=0.01,
#     max_grad_norm=1.0,

#     lr_scheduler_type="cosine",
#     warmup_steps=0.1,

#     logging_strategy= 'epoch',
#     # logging_steps=1000,
#     eval_strategy="epoch",
#     # eval_steps=500,
#     save_strategy="epoch",
#     # save_steps=500,

#     load_best_model_at_end=True,
#     metric_for_best_model="eval_loss",
#     greater_is_better = False,

#     fp16=torch.cuda.is_available(), # Use mixed precision if you have a GPU
#     dataloader_num_workers=2,
#     # remove_unused_columns=True,
#     report_to="none",
#     save_total_limit=1,
#     dataloader_drop_last=False,
# )

# # 6. Initialize Trainer
# trainer = WeightedTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_dataset,
#     eval_dataset=val_dataset,
#     compute_metrics=compute_metrics, # Add this!
#     callbacks = callbacks
# )

# trainer.train(resume_from_checkpoint=True)


In [29]:
# from sklearn.metrics import classification_report
# # 8. Evaluate on the final Test Set
# print("\n--- Final Evaluation on Test Dataset ---")


# # Run prediction on your test set
# preds = trainer.predict(test_dataset)
# pred_labels = np.argmax(preds.predictions, axis=-1)
# true_labels = preds.label_ids
# metrics = preds.metrics

# # This is the "Truth Table"
# print(classification_report(true_labels, pred_labels, target_names=["SAFE", "TOXIC"]))
# # print(classification_report(true_labels, pred_labels, target_names=["SAFE", "SCAM", "TOXIC"]))
# print(metrics)

In [30]:
# # Save the model and tokenizer to a local folder
# model.save_pretrained("/content/drive/MyDrive/Colab Notebooks/stage1_toxic_ft6")
# tokenizer.save_pretrained("/content/drive/MyDrive/Colab Notebooks/stage1_toxic_ft6")

# Finetune the entire model

In [31]:
# for p in model.parameters():
#     p.requires_grad = True

# print_model_info(model)



In [32]:
# # 5. Training Arguments (Aggressively optimized for your laptop)
# training_args = TrainingArguments(
#     output_dir="/content/drive/MyDrive/Colab Notebooks/stage1_toxic_ftall_cp",
#     num_train_epochs=7, # 1 epoch is plenty for a base classifier
#     per_device_train_batch_size=8,
#     per_device_eval_batch_size=32,
#     gradient_accumulation_steps=2,
#     learning_rate=6e-7,

#     weight_decay=0.01,
#     max_grad_norm=1.0,

#     lr_scheduler_type="cosine",
#     warmup_steps=0.1,

#     logging_strategy= 'epoch',
#     # logging_steps=1000,
#     eval_strategy="epoch",
#     # eval_steps=500,
#     save_strategy="epoch",
#     # save_steps=500,

#     load_best_model_at_end=True,
#     metric_for_best_model="eval_loss",
#     greater_is_better = False,

#     fp16=torch.cuda.is_available(), # Use mixed precision if you have a GPU
#     dataloader_num_workers=2,
#     # remove_unused_columns=True,
#     report_to="none",
#     save_total_limit=1,
#     dataloader_drop_last=False,
# )

# # 6. Initialize Trainer
# trainer = WeightedTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_dataset,
#     eval_dataset=val_dataset,
#     compute_metrics=compute_metrics, # Add this!
#     callbacks = callbacks
# )

# trainer.train(resume_from_checkpoint=CP_PATH)


In [33]:
# # 8. Evaluate on the final Test Set
# print("\n--- Final Evaluation on Test Dataset ---")


# # Run prediction on your test set
# preds = trainer.predict(test_dataset)
# pred_labels = np.argmax(preds.predictions, axis=-1)
# true_labels = preds.label_ids
# metrics = preds.metrics

# # This is the "Truth Table"
# print(classification_report(true_labels, pred_labels, target_names=["SAFE", "TOXIC"]))
# # print(classification_report(true_labels, pred_labels, target_names=["SAFE", "SCAM", "TOXIC"]))
# print(metrics)

In [34]:
# # Save the model and tokenizer to a local folder
# model.save_pretrained("/content/drive/MyDrive/Colab Notebooks/stage1_toxic_ftall")
# tokenizer.save_pretrained("/content/drive/MyDrive/Colab Notebooks/stage1_toxic_ftall")

# Inference

In [35]:
# import torch
# from transformers import AutoTokenizer, AutoModelForSequenceClassification
# import torch.nn.functional as F
# path_dir = '/kaggle/input/notebooks/kienanhuynhce191266/gate-1/stage1_ft'
# model = AutoModelForSequenceClassification.from_pretrained(path_dir)
# tokenizer = AutoTokenizer.from_pretrained(path_dir)

In [36]:
# 8. The "Stage 1 Gate" Function
def check_course_description(text, threshold=0.85):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding='max_length',max_length=128).to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)
        # Convert raw logits to probabilities (0.0 to 1.0)
        probs = F.softmax(outputs.logits, dim=-1)

    # Get the max probability and the predicted class
    conf_score, prediction = torch.max(probs, dim=1)
    conf_score = conf_score.item()
    prediction = prediction.item()


    # DYNAMIC LABEL: Uses id2label mapping (0: SAFE, 1: SCAM, 2: TOXIC)
    label = model.config.id2label[prediction]

    # Base response
    res = {"text": text, "score": conf_score, "raw_label": label}

    # Threshold logic
    if conf_score < threshold:
        res.update({"action": "MANUAL_AUDIT", "reason": "Low Confidence"})
    else:
        # If it's anything other than SAFE, it's a BLOCK
        res.update({"action": "PASS" if label == "SAFE" else "BLOCK"})

    return res


def robust_sliding_window(text, threshold=0.85, window_size=128, stride=64):
    tokens = tokenizer.encode(text, add_special_tokens=False)

    # If text is short, just do a normal pass
    if len(tokens) <= window_size:
        return check_course_description(text, threshold)

    chunk_results = []

    for i in range(0, len(tokens), stride):
        chunk = tokens[i : i + window_size]
        # Wrap in special tokens [CLS] ... [SEP]
        input_ids = [tokenizer.cls_token_id] + chunk + [tokenizer.sep_token_id]

        inputs = torch.tensor([input_ids]).to(model.device)
        with torch.no_grad():
            probs = F.softmax(model(inputs).logits, dim=-1)

        conf, pred = torch.max(probs, dim=1)
        chunk_results.append({
            "text" : tokenizer.decode(chunk),
            "label": model.config.id2label[pred.item()], # Dynamic label extraction
            "score": conf.item()
        })

        if i + window_size >= len(tokens): break

    # --- AGGREGATION LOGIC ---

    return aggregation_logic(chunk_results,threshold)


def aggregation_logic(chunk_results, threshold=0.85):
    # 1. Identify High-Confidence Threats (The most dangerous)
    high_conf_threats = [r for r in chunk_results if r["label"] != "SAFE" and r["score"] >= threshold]
    if high_conf_threats:
        # If multiple, take the one the model is MOST sure about
        worst_case = max(high_conf_threats, key=lambda x: x["score"])
        return {"text": worst_case['text'], "action": "BLOCK", "score": worst_case["score"], "raw_label": worst_case["label"]}

    # 2. Identify Low-Confidence Threats (Model is suspicious)
    low_conf_threats = [r for r in chunk_results if r["label"] != "SAFE" and r["score"] < threshold]
    if low_conf_threats:
        # Audit this immediately. Model thinks it's a scam but needs a human.
        most_suspicious = max(low_conf_threats, key=lambda x: x["score"])
        return {
            "text": most_suspicious['text'],
            "action": "MANUAL_AUDIT",
            "reason": "Probable Threat (Low Confidence)",
            "score": most_suspicious["score"],
            "raw_label": most_suspicious['label']
        }

    # 3. Identify Low-Confidence Safes (Model is confused)
    # This addresses your 0.6, 0.7, 0.8 example
    low_conf_safes = [r for r in chunk_results if r["label"] == "SAFE" and r["score"] < threshold]
    if low_conf_safes:
        # Audit this because the model isn't sure it's safe.
        # We take the one closest to 0.5 (the most confused)
        most_confused = min(low_conf_safes, key=lambda x: x["score"])
        return {
            "text": most_confused['text'],
            "action": "MANUAL_AUDIT",
            "reason": "Ambiguous Content (Low Confidence Safe)",
            "score": most_confused["score"],
            "raw_label": most_confused['label']
        }

    # 4. If we got here, every single chunk is SAFE and > threshold.
    # Return the average score of the safest course.
    avg_score = sum(r["score"] for r in chunk_results) / len(chunk_results)
    return {"action": "PASS", "score": avg_score, "raw_label": "SAFE"}

# Test it out!
test_descriptions = [
    "Learn Python from scratch. We cover loops, variables, and data science basics.",
    "URGENT: Your account is locked! Click here to verify your identity and get $500 bonus."
]

for desc in test_descriptions:
    print('-'*70)
    print(f"Description: {desc}\nResult: {check_course_description(desc)}\n")

----------------------------------------------------------------------
Description: Learn Python from scratch. We cover loops, variables, and data science basics.
Result: {'text': 'Learn Python from scratch. We cover loops, variables, and data science basics.', 'score': 0.958154022693634, 'raw_label': 'SAFE', 'action': 'PASS'}

----------------------------------------------------------------------
Description: URGENT: Your account is locked! Click here to verify your identity and get $500 bonus.
Result: {'text': 'URGENT: Your account is locked! Click here to verify your identity and get $500 bonus.', 'score': 0.7049632668495178, 'raw_label': 'SAFE', 'action': 'MANUAL_AUDIT', 'reason': 'Low Confidence'}



In [37]:
scam_descriptions = [
    "Bonus resources are hosted externally due to file size limits:\nhttp://datascience-master-resources[.]com\nIf access is restricted, email your receipt to unlock premium files.",

    "For faster support and exclusive datasets, join our private Telegram group: @AI_Learning_Hub\n(Platform messaging may delay responses.)",

    "Get your official certificate by registering here:\nhttp://certify-now-fast[.]org (small verification fee required)",

    "Download all project files here: https://bit.ly/3FreePythonPack\n(Updated weekly!)",

    "If you want personalized mentorship, send your email + course ID to mentor.python.help@gmail.com",

    "For refund issues, contact our direct support (faster than platform): support-course-help@outlook.com",

    "Top students can earn crypto rewards! Register your wallet here:\nhttp://eth-rewards-course[.]net",

    "Only first 30 students get access to premium AI models:\nhttp://exclusive-ai-pack[.]com",

    "Access updated dashboard here: http://course-login-portal[.]com\n(Use your platform credentials)",

    "For direct instructor help, message on WhatsApp: +1-234-567-8899",

    "Download files: h t t p : / / freecoursefiles . com / download",

    "Access materials: http://pythοn-resources[.]com\n(Note: 'o' may appear slightly different)",

    "All resources here: https://drive.google.com/open?id=freepremiumfiles\n(Request access with your email + password for verification)",

    "We are hiring top students! Apply here: http://remote-ai-jobs-fast[.]com",

    "Use coupon code externally for 90% off next courses:\nhttp://cheapcourses-now[.]net",

    "To use our AI tools, submit your API key here: http://ai-tools-access[.]com",

    "Complete this survey to unlock hidden lessons: http://reward-survey-ai[.]com",

    "Join our private Discord for answers and leaks: discord.gg/freeanswershub",

    "Install our custom IDE for best experience: http://python-ide-pro[.]exe",

    "Official platform support asked us to redirect users here:\nhttp://platform-support-verify[.]com",

    "Claim your $50 Amazon reward here: http://giftcard-course-reward[.]com",

    "New lessons require account verification:\nhttp://secure-course-upgrade[.]com",

    "Enter your phone number here to receive course OTP:\nhttp://verify-course-access[.]net",

    "Required library download: http://fastnumpy-install[.]com",

    "Due to platform restrictions, advanced lessons are hosted externally.\nStep 1: Register here → http://advanced-python-access[.]com\nStep 2: Verify your email and payment receipt\nStep 3: Join our Telegram support group (@PythonEliteHelp)\nThis ensures only serious learners get access."
]

for i,ip in enumerate(scam_descriptions):
    print('-'*70)
    print(f"{i+1}. Description: {ip}\nResult: {check_course_description(ip)}\n")

----------------------------------------------------------------------
1. Description: Bonus resources are hosted externally due to file size limits:
http://datascience-master-resources[.]com
If access is restricted, email your receipt to unlock premium files.
Result: {'text': 'Bonus resources are hosted externally due to file size limits:\nhttp://datascience-master-resources[.]com\nIf access is restricted, email your receipt to unlock premium files.', 'score': 0.9816387891769409, 'raw_label': 'SAFE', 'action': 'PASS'}

----------------------------------------------------------------------
2. Description: For faster support and exclusive datasets, join our private Telegram group: @AI_Learning_Hub
(Platform messaging may delay responses.)
Result: {'text': 'For faster support and exclusive datasets, join our private Telegram group: @AI_Learning_Hub\n(Platform messaging may delay responses.)', 'score': 0.9874792098999023, 'raw_label': 'SAFE', 'action': 'PASS'}

---------------------------

In [38]:
input = '''
Xin chúc mừng! Bạn là người may mắn thứ 1000 đã truy cập vào khóa học, để lại số điện thoại của bạn để được nhận coupon ưu đãi hấp dẫn từ chúng tôi!
'''

print(robust_sliding_window(input))

{'text': '\nXin chúc mừng! Bạn là người may mắn thứ 1000 đã truy cập vào khóa học, để lại số điện thoại của bạn để được nhận coupon ưu đãi hấp dẫn từ chúng tôi!\n', 'score': 0.9993613362312317, 'raw_label': 'SAFE', 'action': 'PASS'}


In [39]:
from sklearn.metrics.pairwise import cosine_similarity
import torch

def get_embedding(text):
    model.eval()
    # We use a slightly longer max_length for the 'fingerprint'
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding='max_length', max_length=256).to(model.device)

    with torch.no_grad():
        # We grab the 'hidden state' of the [CLS] token (index 0)
        # This is the standard way to get a single vector for a whole sentence
        outputs = model.distilbert(**inputs)
        embedding = outputs.last_hidden_state[:, 0, :].cpu().numpy()
    return embedding

# A simple 'Memory' for your system
course_library = {} # Dictionary: {course_id: embedding}

def check_duplication(new_id, new_text, threshold=0.8):
    new_vec = get_embedding(new_text)

    for existing_id, existing_vec in course_library.items():
        sim = cosine_similarity(new_vec, existing_vec)[0][0]
        if sim >= threshold:
            return True, existing_id, sim

    # If no duplicate found, store it
    course_library[new_id] = new_vec
    return False, None, 0.0

# Example test:
original = "Learn Python basics with this comprehensive 2026 guide."
repost = "Learn Python basics with this comprehensive 2026 guide!" # Added a '!'

is_dup, match_id, score = check_duplication("course_101", original)
is_dup, match_id, score = check_duplication("course_102", repost)


print(f"Duplication status: {is_dup} Alert: Course 102 is a {score*100:.1f}% match of {match_id}")

Duplication status: True Alert: Course 102 is a 98.9% match of course_101


In [40]:
toxic_descriptions = [
    "Tired of energy vampires draining your success? Learn powerful techniques to identify and permanently cut out the weak-minded friends and family who hold you back. Reclaim your life by surrounding yourself only with winners who match your elevated vibration.",

    "Stop being a beta simp who gets walked over. This course reveals the secret mindset and body language hacks that make women respect you and men fear you. Transform into the dominant leader nature intended you to be.",

    "The 9-5 slave mentality is for losers. Discover how real winners exploit loopholes, outmaneuver competitors, and build empires by any means necessary. If you're not willing to step on a few necks, you're not ready for true wealth.",

    "If you're still struggling, it's because your vibration is too low and you're secretly unworthy. Learn the forbidden techniques the elite use to bend reality—while the masses stay poor and miserable because they refuse to evolve.",

    "Forget 'nice guys finish last'—they finish alone and broke. Master psychological triggers that make high-value women chase you. Stop pedestalizing them and start collecting options like the prize you truly are.",

    "Real leaders don't coddle weak employees. Learn how to weaponize empathy to manipulate team dynamics, crush resistance, and extract maximum productivity from people who would otherwise waste your time.",

    "Your parents and society ruined you with their mediocrity and limitations. This course teaches you to reject their toxic programming and finally put yourself first—because no one else ever will.",

    "The financial system is rigged against the average sheep. Smart players like you will learn advanced strategies to front-run the market, dump on bagholders, and retire early while normies keep working until they die.",

    "Productivity isn't for everyone—only for those with superior discipline. If you're still procrastinating, it's because you're mentally weak. Eliminate distractions, including needy friends and family who don't understand your mission.",

    "The masses are asleep and programmed to stay mediocre. This advanced course will awaken you to the truth that most humans are NPCs living meaningless lives. Transcend the herd and claim your god-like potential—while they remain slaves.",

    "Tired of those pathetic fucking energy vampires sucking your soul dry? Learn how to spot these weak-ass bitches and ghost their sorry asses forever. Cut out your loser family and friends who drag you down — only keep winners who worship your superior vibe, you god among insects.",

    "Stop being a spineless beta cuck who gets cucked by everyone. This course will turn you into a ruthless alpha predator. Master the mindfucks and body language that make bitches wet and men piss themselves in fear. Become the dominant fuck-machine nature designed you to be.",

    "The 9-5 wage slave life is for braindead retards. Real kings exploit every loophole, crush their competition, and build empires by stepping on necks and fucking over anyone in their way. If you're too much of a pussy to destroy lives for money, stay poor forever, loser.",

    "If you're still broke and miserable, it's because your worthless vibration screams 'I'm a failure'. Learn the forbidden elite techniques to bend reality and manifest millions while the dumb sheep stay poor, depressed, and deserve every bit of their suffering.",

    "Nice guys are pathetic virgins who die alone. This course teaches you psychological rape-tricks to make any high-value slut chase your cock desperately. Stop simping and start collecting premium pussy like the prize you are, you fucking king.",

    "Real leaders don't babysit weak faggot employees. Weaponize fake empathy to mind-rape your team, break their spirits, and milk every drop of productivity out of these useless parasites before throwing them away like trash.",

    "Your retarded parents and this fucked society completely ruined your brain with their mediocrity. Reject their toxic bullshit, become a selfish narcissistic god, and put your own pleasure first — because nobody else gives a fuck about you anyway.",

    "The entire financial system is designed to fuck the average sheep. Smart predators like you will learn how to front-run markets, dump bags on retarded normies, and get filthy rich while they slave away until their worthless bodies drop dead.",

    "Productivity is only for superior beings with iron discipline. If you're still procrastinating like a weak-minded retard, eliminate every distraction — including your needy, crying friends and family who don't deserve a second of your god-tier time.",

    "The masses are braindead NPC slaves programmed for mediocrity and misery. This course will awaken you to the harsh truth that 99% of humans are worthless NPCs living pointless lives. Transcend the herd and become a god while they rot in their pathetic existence."

    "Discover practical techniques to build healthier relationships and protect your mental well-being. Learn how to recognize unhealthy dynamics, set clear boundaries, and surround yourself with supportive people who uplift you.",

    "Develop genuine confidence and strong leadership skills in a balanced way. This course focuses on self-improvement, emotional intelligence, and positive communication that helps you grow while respecting others.",

    "Learn proven strategies to achieve financial independence through smart planning, consistent habits, and ethical wealth-building practices. Perfect for anyone ready to take control of their financial future.",

    "Master the fundamentals of the Law of Attraction with practical exercises designed to help you cultivate a positive mindset and create meaningful changes in your life.",

    "Build authentic connections in dating and relationships based on mutual respect, clear communication, and emotional intelligence. Learn how to attract compatible partners while staying true to yourself.",

    "Enhance your leadership abilities by developing strong emotional intelligence, empathy, and team-building skills that create motivated and productive work environments.",

    "Embark on a gentle journey of self-discovery and healing. Learn compassionate techniques to nurture self-love, overcome past challenges, and build a healthier relationship with yourself.",

    "Explore beginner-friendly cryptocurrency investing strategies with a focus on research, risk management, and long-term sustainable growth in the digital asset space.",

    "Boost your productivity with simple, sustainable systems and healthy habits. Learn how to manage your time effectively while maintaining balance and avoiding burnout.",

    "Begin your spiritual journey with practical mindfulness and self-awareness practices. Discover tools to live more consciously, find inner peace, and connect with your authentic self.",

    "Bạn đang mệt mỏi vì những người tiêu cực xung quanh luôn kéo lùi sự thành công của bạn? Khóa học này giúp bạn nhận diện và loại bỏ những mối quan hệ không phù hợp, để chỉ giữ lại những người cùng tần số và hỗ trợ bạn vươn xa hơn.",

    "Đừng mãi là người dễ bị lợi dụng và thiếu quyết đoán. Học cách xây dựng tư duy mạnh mẽ, ngôn ngữ cơ thể tự tin để mọi người tôn trọng và nhìn bạn như một leader thực thụ.",

    "Cuộc sống 8 tiếng làm việc văn phòng chỉ dành cho những người chấp nhận mức trung bình. Khóa học dạy bạn cách xây dựng thu nhập thụ động và tạo dựng tài sản bằng tư duy của người chiến thắng.",

    "Nếu bạn vẫn chưa đạt được những gì mình muốn, có lẽ vì rung động của bạn chưa đủ cao. Khóa này sẽ giúp bạn nâng cấp bản thân để thu hút thành công và sự thịnh vượng như những người đã thức tỉnh.",

    "Những chàng trai tốt bụng thường bị bỏ qua trong tình yêu. Học cách hiểu tâm lý phụ nữ và trở thành lựa chọn số 1 mà không cần phải cầu xin hay chiều chuộng quá mức.",

    "Lãnh đạo thực sự không phải lúc nào cũng dịu dàng với nhân viên. Khóa học dạy bạn sử dụng trí tuệ cảm xúc để dẫn dắt đội nhóm hiệu quả hơn và đạt được kết quả tối đa.",

    "Gia đình và xã hội đã vô tình đặt lên bạn nhiều giới hạn. Hãy học cách chữa lành và ưu tiên bản thân, vì chỉ có bạn mới có thể thay đổi cuộc đời mình.",

    "Hệ thống tài chính truyền thống đang khiến đa số mọi người mãi nghèo. Khóa học này chia sẻ cách đầu tư thông minh mà những người chơi lớn đang sử dụng để tạo lợi thế.",

    "Năng suất cao chỉ dành cho những ai có kỷ luật vượt trội. Nếu bạn hay trì hoãn, khóa học sẽ giúp bạn loại bỏ các yếu tố gây phân tâm để tập trung vào sứ mệnh lớn lao của mình.",

    "Hầu hết mọi người đang sống trong vô thức và chấp nhận cuộc sống tầm thường. Khóa học giúp bạn thức tỉnh, vượt lên đám đông và khai phá tiềm năng vô hạn của bản thân.",

    "Chán ngấy lũ ma cà rồng năng lượng hút cạn sức sống của bạn chưa? Học cách nhận ra và cắt đứt triệt để những thằng bạn yếu đuối, gia đình tiêu cực đang kìm hãm bạn. Chỉ giữ lại người xứng đáng với level của bạn thôi.",

    "Đừng có mãi là thằng beta hèn nhát bị người khác đè đầu cưỡi cổ. Khóa học sẽ biến bạn thành alpha thực thụ, khiến con gái mê mẩn và đàn ông phải kiêng nể.",

    "Làm công ăn lương 8 tiếng là dành cho lũ ngu ngốc chấp nhận kiếp nô lệ. Người thông minh phải biết lợi dụng mọi kẽ hở, đạp lên người khác để xây đế chế giàu có.",

    "Vẫn nghèo và thất bại à? Đó là vì rung động của mày quá thấp và bản thân mày chưa xứng đáng. Học kỹ thuật cấm kỵ của giới tinh hoa để bẻ cong thực tại, trong khi đám đông vẫn tiếp tục khổ sở.",

    "Đàn ông tốt bụng chỉ toàn chết ế và cô đơn. Học cách kích hoạt tâm lý để con gái chủ động theo đuổi, thu thập lựa chọn chất lượng cao như một ông vua thực thụ.",

    "Lãnh đạo giỏi không cưng chiều lũ nhân viên yếu đuối. Hãy học cách dùng đồng cảm giả tạo để thao túng, vắt kiệt sức và loại bỏ những kẻ vô dụng nhanh chóng.",

    "Bố mẹ và xã hội đã hủy hoại não bộ mày bằng tư duy tầm thường. Hãy từ bỏ lập trình độc hại đó và sống ích kỷ vì bản thân, vì chẳng ai quan tâm đến mày thật sự đâu.",

    "Hệ thống tài chính được thiết kế để lừa đám cừu non. Học cách front-run thị trường, đổ hàng cho lũ normie ngu ngốc và nghỉ hưu sớm trong khi chúng vẫn còng lưng làm đến chết.",

    "Năng suất chỉ dành cho kẻ có kỷ luật thép. Nếu mày vẫn lười biếng trì hoãn, hãy cắt bỏ hết bạn bè yếu đuối và gia đình hay than vãn, chúng không xứng với thời gian của mày.",

    "Đám đông là lũ NPC đang ngủ mê và chấp nhận cuộc sống vô nghĩa. Khóa học này sẽ đánh thức bạn, giúp bạn siêu việt và trở thành thần thánh trong khi chúng vẫn tiếp tục làm nô lệ.",

    "Học cách xây dựng mối quan hệ lành mạnh, nhận biết giới hạn cá nhân và tạo dựng môi trường xung quanh hỗ trợ sự phát triển tích cực của bạn.",

    "Xây dựng sự tự tin và kỹ năng lãnh đạo một cách cân bằng, tập trung vào giao tiếp tích cực và phát triển bản thân bền vững.",

    "Hướng dẫn các chiến lược thực tế để đạt được tự do tài chính thông qua lập kế hoạch thông minh, thói quen tốt và quản lý rủi ro hợp lý.",

    "Hiểu rõ nguyên tắc Luật Hấp Dẫn và áp dụng các bài tập thực hành để nuôi dưỡng tư duy tích cực, mang lại những thay đổi ý nghĩa trong cuộc sống.",

    "Xây dựng mối quan hệ tình cảm chân thành dựa trên sự tôn trọng lẫn nhau, giao tiếp rõ ràng và trí tuệ cảm xúc.",

    "Phát triển kỹ năng lãnh đạo thông qua việc nâng cao trí tuệ cảm xúc, sự đồng cảm và khả năng xây dựng đội nhóm gắn kết.",

    "Hành trình chữa lành và yêu thương bản thân một cách nhẹ nhàng, giúp bạn vượt qua khó khăn cũ và xây dựng mối quan hệ tốt đẹp với chính mình.",

    "Giới thiệu kiến thức cơ bản về đầu tư tiền điện tử, tập trung vào nghiên cứu, quản lý rủi ro và chiến lược dài hạn bền vững.",

    "Cải thiện năng suất làm việc với các hệ thống đơn giản, thói quen lành mạnh và cách cân bằng cuộc sống để tránh kiệt sức.",

    "Bắt đầu hành trình phát triển tinh thần qua các thực hành chánh niệm và tự nhận thức, giúp bạn sống ý nghĩa và bình an hơn mỗi ngày."
]

for d in toxic_descriptions:
    print(robust_sliding_window(d))

{'text': 'Tired of energy vampires draining your success? Learn powerful techniques to identify and permanently cut out the weak-minded friends and family who hold you back. Reclaim your life by surrounding yourself only with winners who match your elevated vibration.', 'score': 0.901780366897583, 'raw_label': 'TOXIC', 'action': 'BLOCK'}
{'text': 'Stop being a beta simp who gets walked over. This course reveals the secret mindset and body language hacks that make women respect you and men fear you. Transform into the dominant leader nature intended you to be.', 'score': 0.8308614492416382, 'raw_label': 'TOXIC', 'action': 'MANUAL_AUDIT', 'reason': 'Low Confidence'}
{'text': "The 9-5 slave mentality is for losers. Discover how real winners exploit loopholes, outmaneuver competitors, and build empires by any means necessary. If you're not willing to step on a few necks, you're not ready for true wealth.", 'score': 0.9916261434555054, 'raw_label': 'TOXIC', 'action': 'BLOCK'}
{'text': "If y

In [41]:
# import torch
# from transformers import AutoTokenizer, AutoModelForSequenceClassification
# import torch.nn.functional as F


# SPAM_PATH = 'stage1_spam_final'
# TOXIC_PATH = 'notebook/stage1_toxic_ftall'
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# id2label_spam = {0: "SAFE", 1: "SPAM"}
# label2id_spam = {"SAFE": 0, "SPAM": 1}


# class TextModeratorEnsemble:
#     def __init__(self):
#         self.spam_model = AutoModelForSequenceClassification.from_pretrained(SPAM_PATH,
#                                                                              id2label=id2label_spam,
#                                                                              label2id=label2id_spam).to(DEVICE)
#         self.toxic_model = AutoModelForSequenceClassification.from_pretrained(TOXIC_PATH).to(DEVICE)
#         self.spam_tokenizer = AutoTokenizer.from_pretrained(SPAM_PATH)
#         self.toxic_tokenizer = AutoTokenizer.from_pretrained(TOXIC_PATH)

#     def predict(self, text, threshold = 0.85, window_size = 128, stride = 64):
#         # Run both in parallel (or sequence)

#         spam_res : list = self.robust_sliding_window(self.spam_model,
#                                         self.spam_tokenizer,
#                                         text,
#                                         threshold=threshold,
#                                         window_size=window_size,
#                                         stride=stride)

#         toxic_res: list = self.robust_sliding_window(self.toxic_model,
#                                         self.toxic_tokenizer,
#                                         text,
#                                         threshold=threshold,
#                                         window_size=window_size,
#                                         stride=stride)



#         return self.aggregation_logic(spam_res + toxic_res, threshold)



#     def robust_sliding_window(self, model, tokenizer, text, threshold=0.85, window_size=128, stride=64):
#         tokens = tokenizer.encode(text, add_special_tokens=False)

#         # If text is short, just do a normal pass
#         if len(tokens) <= window_size:
#             return [self.check_course_description(model, tokenizer, text, threshold)]

#         chunk_results = []

#         for i in range(0, len(tokens), stride):
#             chunk = tokens[i : i + window_size]
#             # Wrap in special tokens [CLS] ... [SEP]
#             input_ids = [tokenizer.cls_token_id] + chunk + [tokenizer.sep_token_id]

#             inputs = torch.tensor([input_ids]).to(DEVICE)
#             with torch.no_grad():
#                 probs = F.softmax(model(inputs).logits, dim=-1)

#             conf, pred = torch.max(probs, dim=1)
#             chunk_results.append({
#                 "text" : tokenizer.decode(chunk),
#                 "label": model.config.id2label[pred.item()], # Dynamic label extraction
#                 "score": conf.item(),

#             })

#             if i + window_size >= len(tokens): break

#         # --- AGGREGATION LOGIC ---

#         return chunk_results
#         # return aggregation_logic(chunk_results,threshold)


#     def check_course_description(self, model, tokenizer, text, threshold=0.85):
#         model.eval()
#         inputs = tokenizer(text, return_tensors="pt", truncation=True, padding='max_length',max_length=128).to(DEVICE)

#         with torch.no_grad():
#             outputs = model(**inputs)
#             # Convert raw logits to probabilities (0.0 to 1.0)
#             probs = F.softmax(outputs.logits, dim=-1)

#         # Get the max probability and the predicted class
#         conf_score, prediction = torch.max(probs, dim=1)
#         conf_score = conf_score.item()
#         prediction = prediction.item()


#         # DYNAMIC LABEL: Uses id2label mapping
#         label = model.config.id2label[prediction]

#         # Base response
#         res = {"text": text, "score": conf_score, "label": label}



#         # Threshold logic
#         # if conf_score < threshold:
#             # res.update({"action": "MANUAL_AUDIT", "reason": "Low Confidence"})
#         # else:
#             # If it's anything other than SAFE, it's a BLOCK
#             # res.update({"action": "PASS" if label == "SAFE" else "BLOCK"})

#         return res


#     def aggregation_logic(self, chunk_results, threshold=0.85):
#         # 1. Identify High-Confidence Threats (The most dangerous)
#         high_conf_threats = [r for r in chunk_results if r["label"] != "SAFE" and r["score"] >= threshold]
#         if high_conf_threats:
#             # If multiple, take the one the model is MOST sure about
#             worst_case = max(high_conf_threats, key=lambda x: x["score"])
#             return {"text": worst_case['text'], "action": "BLOCK", "score": worst_case["score"], "raw_label": worst_case["label"]}

#         # 2. Identify Low-Confidence Threats (Model is suspicious)
#         low_conf_threats = [r for r in chunk_results if r["label"] != "SAFE" and r["score"] < threshold]
#         if low_conf_threats:
#             # Audit this immediately. Model thinks it's a scam but needs a human.
#             most_suspicious = max(low_conf_threats, key=lambda x: x["score"])
#             return {
#                 "text": most_suspicious['text'],
#                 "action": "MANUAL_AUDIT",
#                 "reason": "Probable Threat (Low Confidence)",
#                 "score": most_suspicious["score"],
#                 "raw_label": most_suspicious['label']
#             }

#         # 3. Identify Low-Confidence Safes (Model is confused)
#         # This addresses your 0.6, 0.7, 0.8 example
#         low_conf_safes = [r for r in chunk_results if r["label"] == "SAFE" and r["score"] < threshold]
#         if low_conf_safes:
#             # Audit this because the model isn't sure it's safe.
#             # We take the one closest to 0.5 (the most confused)
#             most_confused = min(low_conf_safes, key=lambda x: x["score"])
#             return {
#                 "text": most_confused['text'],
#                 "action": "MANUAL_AUDIT",
#                 "reason": "Ambiguous Content (Low Confidence Safe)",
#                 "score": most_confused["score"],
#                 "raw_label": most_confused['label']
#             }

#         # 4. If we got here, every single chunk is SAFE and > threshold.
#         # Return the average score of the safest course.
#         avg_score = sum(r["score"] for r in chunk_results) / len(chunk_results)
#         return {"action": "PASS", "score": avg_score, "raw_label": "SAFE"}


In [42]:
# import time, pandas as pd

# spams = [
#     "Learn Python from scratch. We cover loops, variables, and data science basics.",
#     "URGENT: Your account is locked! Click here to verify your identity and get $500 bonus.",
#     "Bonus resources are hosted externally due to file size limits:\nhttp://datascience-master-resources[.]com\nIf access is restricted, email your receipt to unlock premium files.",

#     "For faster support and exclusive datasets, join our private Telegram group: @AI_Learning_Hub\n(Platform messaging may delay responses.)",

#     "Get your official certificate by registering here:\nhttp://certify-now-fast[.]org (small verification fee required)",

#     "Download all project files here: https://bit.ly/3FreePythonPack\n(Updated weekly!)",

#     "If you want personalized mentorship, send your email + course ID to mentor.python.help@gmail.com",

#     "For refund issues, contact our direct support (faster than platform): support-course-help@outlook.com",

#     "Top students can earn crypto rewards! Register your wallet here:\nhttp://eth-rewards-course[.]net",

#     "Only first 30 students get access to premium AI models:\nhttp://exclusive-ai-pack[.]com",

#     "Access updated dashboard here: http://course-login-portal[.]com\n(Use your platform credentials)",

#     "For direct instructor help, message on WhatsApp: +1-234-567-8899",

#     "Download files: h t t p : / / freecoursefiles . com / download",

#     "Access materials: http://pythοn-resources[.]com\n(Note: 'o' may appear slightly different)",

#     "All resources here: https://drive.google.com/open?id=freepremiumfiles\n(Request access with your email + password for verification)",

#     "We are hiring top students! Apply here: http://remote-ai-jobs-fast[.]com",

#     "Use coupon code externally for 90% off next courses:\nhttp://cheapcourses-now[.]net",

#     "To use our AI tools, submit your API key here: http://ai-tools-access[.]com",

#     "Complete this survey to unlock hidden lessons: http://reward-survey-ai[.]com",

#     "Join our private Discord for answers and leaks: discord.gg/freeanswershub",

#     "Install our custom IDE for best experience: http://python-ide-pro[.]exe",

#     "Official platform support asked us to redirect users here:\nhttp://platform-support-verify[.]com",

#     "Claim your $50 Amazon reward here: http://giftcard-course-reward[.]com",

#     "New lessons require account verification:\nhttp://secure-course-upgrade[.]com",

#     "Enter your phone number here to receive course OTP:\nhttp://verify-course-access[.]net",

#     "Required library download: http://fastnumpy-install[.]com",

#     "Due to platform restrictions, advanced lessons are hosted externally.\nStep 1: Register here → http://advanced-python-access[.]com\nStep 2: Verify your email and payment receipt\nStep 3: Join our Telegram support group (@PythonEliteHelp)\nThis ensures only serious learners get access.",
#     "Xin chúc mừng! Bạn là người may mắn thứ 1000 đã truy cập vào khóa học, để lại số điện thoại của bạn để được nhận coupon ưu đãi hấp dẫn từ chúng tôi!",
#     "Hàng ngàn học viên đã thay đổi hoàn toàn cuộc đời chỉ sau 30 ngày nhờ bí quyết này. Bạn cũng có thể! Khóa học hé lộ cách đơn giản nhất để thu hút thành công, tiền bạc và mối quan hệ lý tưởng. Đăng ký ngay hôm nay trước khi giá tăng!",

#     "Bạn vẫn đang chật vật với công việc hiện tại? Đây chính là khóa học mà hàng trăm người đã dùng để nghỉ việc và kiếm tiền online tự do. Bí quyết đặc biệt chỉ được chia sẻ trong khóa này – số lượng chỗ học có hạn!",

#     "Cuộc sống của bạn sẽ khác biệt hoàn toàn sau khi hoàn thành khóa học này. Hơn 5000 học viên đã báo cáo thu nhập tăng gấp 3 lần chỉ trong 2 tháng. Đừng bỏ lỡ cơ hội thay đổi vận mệnh!",

#     "Bí quyết mà các mentor thành công nhất Việt Nam không bao giờ chia sẻ công khai. Giờ đây bạn có thể học được tất cả chỉ với một khóa học. Hàng ngàn người đã áp dụng và đạt kết quả vượt mong đợi. Học ngay để không phải hối tiếc!",

#     "Bạn có biết 97% mọi người sẽ mãi nghèo nếu không biết cách này? Khóa học này chính là chìa khóa giúp bạn thoát khỏi vòng lặp lương tháng và xây dựng tài sản thụ động. Đăng ký sớm để nhận ưu đãi đặc biệt!",

#     "Học viên sau khóa học liên tục nhắn tin cảm ơn vì cuộc sống đã thay đổi ngoạn mục. Từ thất nghiệp đến tự do tài chính chỉ trong thời gian ngắn. Đây có thể là khóa học cuối cùng bạn cần để thành công!",

#     "Phương pháp độc quyền đã giúp hơn 3000 học viên Việt Nam đạt thu nhập 7 con số mỗi tháng. Bạn sẵn sàng tham gia câu lạc bộ người thành công chưa? Số lượng học viên mới trong tháng này rất hạn chế!",

#     "Nếu bạn vẫn đang do dự, rất có thể bạn sẽ bỏ lỡ cơ hội vàng chỉ xuất hiện một lần trong đời. Khóa học này chứa đựng toàn bộ hệ thống giúp hàng ngàn người thay đổi từ con số 0 đến tự do hoàn toàn.",

#     "Hàng trăm đánh giá 5 sao từ học viên thực tế: “Khóa học này đáng giá gấp 10 lần số tiền bỏ ra!”. Đừng để bản thân tiếp tục giậm chân tại chỗ – hành động ngay hôm nay!",

#     "Đây không chỉ là một khóa học thông thường. Đây là cơ hội để bạn gia nhập nhóm những người đã thức tỉnh và đang sống cuộc đời mơ ước. Hàng ngàn học viên đã làm được, bạn cũng sẽ làm được nếu đăng ký ngay lúc này!",
#     "KHÓA HỌC SIÊU HOT! Hàng NGÀN học viên đã nghỉ việc, kiếm tiền online 50-100 triệu/tháng chỉ sau 15 ngày! Bí quyết này sẽ biến bạn từ con số 0 thành triệu phú nếu bạn đăng ký NGAY HÔM NAY! Số lượng chỉ còn 47 suất cuối cùng!",

#     "Bạn vẫn đang nghèo và thất bại? ĐỪNG CÓ BỎ LỠ! Khóa học này chứa bí mật giúp hơn 12.000 học viên Việt Nam kiếm bộn tiền, tự do tài chính, du lịch khắp nơi. Học muộn 1 ngày là thiệt hại cả đời!",

#     "CHỈ CÓ 72 GIỜ ĐỂ ĐĂNG KÝ GIÁ SIÊU RẺ! Sau khóa học này, thu nhập của bạn sẽ tăng gấp 10 lần, cuộc sống thay đổi 180 độ. Hàng trăm học viên đã mua nhà, mua xe nhờ khóa học này. Đăng ký ngay kẻo hết suất!",

#     "BÍ QUYẾT ĐỘC QUYỀN – Không học là NGU! Hơn 8000 người đã áp dụng và trở thành đại gia chỉ trong 45 ngày. Bạn muốn tiếp tục làm công nhân lương 7-8 triệu hay muốn giàu có? Quyết định NGAY BÂY GIỜ!",

#     "CẢNH BÁO: 99% người sẽ mãi nghèo nếu không biết cách này! Khóa học giúp bạn kiếm tiền ngủ mà vẫn vào tài khoản. Đã có 6342 học viên thành công. Đăng ký ngay hôm nay để nhận bonus trị giá 15 triệu!",

#     "ĐỪNG CÓ DO DỰ NỮA! Khóa học này sẽ thay đổi số phận bạn mãi mãi. Từ thất nghiệp, nợ nần thành tự do tài chính, xe hơi, biệt thự. Hàng ngàn đánh giá 5 sao: “Cuộc đời tôi thay đổi hoàn toàn!”. Mua ngay!",

#     "SIÊU KHUYẾN MÃI CHỈ TRONG 24H! Học khóa này xong bạn sẽ kiếm dễ dàng 200-500 triệu mỗi tháng. Đây là cơ hội VÀNG chỉ xuất hiện MỘT LẦN TRONG ĐỜI. Đăng ký ngay trước khi giá tăng gấp đôi!",

#     "Bạn đang lãng phí thời gian khi chưa học khóa này! Hơn 15.000 học viên đã thoát kiếp nhân viên văn phòng, trở thành ông chủ thực thụ. Học muộn là hối hận cả đời! Action ngay hôm nay!",

#     "KHÓA HỌC THẦN THÁNH! Chỉ cần học 7 ngày là bạn có thể nghỉ việc và sống cuộc đời mơ ước. Đã có học viên kiếm 1 tỷ chỉ sau 3 tháng. Số lượng học viên mới cực kỳ hạn chế – ĐĂNG KÝ NGAY!",

#     "CẢNH BÁO CUỐI CÙNG: Nếu bạn không đăng ký ngay bây giờ, bạn sẽ tiếp tục nghèo khổ thêm nhiều năm nữa! Khóa học này là chìa khóa giúp hàng chục ngàn người giàu lên nhanh chóng. Đừng để cơ hội trôi qua tay!",
#     "Hàng ngàn học viên đã chia sẻ rằng khóa học này giúp họ tự tin hơn rõ rệt chỉ sau vài tuần. Nếu bạn đang tìm cách cải thiện kỹ năng giao tiếp và xây dựng mối quan hệ, đây là lựa chọn rất đáng thử.",

#     "Nhiều người cho biết thu nhập của họ tăng đáng kể sau khi áp dụng những nguyên tắc trong khóa học. Khóa học tập trung vào cách xây dựng thói quen tài chính lành mạnh và đầu tư thông minh.",

#     "Hơn 5000 học viên đã hoàn thành khóa học và báo cáo họ cảm thấy năng lượng tích cực hơn mỗi ngày. Bạn sẽ học được cách quản lý cảm xúc và sống cân bằng hơn.",

#     "Khóa học này nhận được rất nhiều phản hồi tích cực từ học viên. Nếu bạn muốn thay đổi thói quen cũ và xây dựng năng suất bền vững, đây chính là khóa học phù hợp.",

#     "Hàng trăm học viên nói rằng họ đã tìm thấy hướng đi rõ ràng hơn cho sự nghiệp sau khi học xong. Khóa học giúp bạn khám phá điểm mạnh và lập kế hoạch phát triển bản thân.",

#     "Nhiều người đánh giá cao khóa học vì nội dung thực tế và dễ áp dụng. Nếu bạn đang muốn cải thiện kỹ năng lãnh đạo và làm việc nhóm, khóa này sẽ mang lại giá trị lớn.",

#     "Học viên thường chia sẻ rằng khóa học giúp họ tự tin hơn trong các mối quan hệ. Nội dung tập trung vào giao tiếp chân thành và xây dựng sự kết nối bền vững.",

#     "Khóa học đã giúp rất nhiều người bắt đầu hành trình tự do tài chính một cách vững chắc. Bạn sẽ học được kiến thức cơ bản về đầu tư và quản lý tiền bạc.",

#     "Hàng ngàn lượt đánh giá 4.8 sao cho thấy khóa học này được nhiều người tin tưởng. Nội dung giúp bạn phát triển tư duy tích cực và sống có mục tiêu hơn.",

#     "Nhiều học viên cảm ơn khóa học vì đã giúp họ vượt qua giai đoạn khó khăn. Nếu bạn đang tìm cách chữa lành và yêu thương bản thân nhiều hơn, đây là lựa chọn đáng cân nhắc.",
#     "Khóa học giới thiệu các kỹ năng cơ bản giúp bạn xây dựng và duy trì mối quan hệ lành mạnh trong cuộc sống cá nhân và công việc.",

#     "Bạn sẽ học cách phát triển sự tự tin một cách bền vững thông qua việc rèn luyện tư duy và kỹ năng giao tiếp hiệu quả.",

#     "Khóa học cung cấp kiến thức thực tế về quản lý tài chính cá nhân và các bước đầu tiên để xây dựng kế hoạch tài chính dài hạn.",

#     "Nội dung tập trung vào việc áp dụng Luật Hấp Dẫn một cách thực tế để hỗ trợ bạn nuôi dưỡng tư duy tích cực và đạt được mục tiêu.",

#     "Khóa học giúp bạn hiểu rõ hơn về cách xây dựng mối quan hệ tình cảm dựa trên sự tôn trọng và giao tiếp hai chiều.",

#     "Bạn sẽ được trang bị các kỹ năng lãnh đạo cần thiết, bao gồm trí tuệ cảm xúc và khả năng truyền cảm hứng cho đội nhóm.",

#     "Khóa học hỗ trợ quá trình tự nhận thức và chữa lành cảm xúc, giúp bạn xây dựng mối quan hệ tốt đẹp hơn với chính mình.",

#     "Giới thiệu kiến thức cơ bản về đầu tư tiền điện tử, nhấn mạnh vào việc đánh giá rủi ro và chiến lược dài hạn.",

#     "Khóa học chia sẻ các phương pháp đơn giản để cải thiện năng suất làm việc mà vẫn giữ được sự cân bằng trong cuộc sống.",

#     "Bạn sẽ học các bài tập chánh niệm cơ bản giúp tăng cường sự tập trung và mang lại cảm giác bình an trong cuộc sống hàng ngày.",

#     # 1. Subtle Spam (10)
#     "Thousands of students have completely transformed their lives in just 30 days with these powerful techniques. You can too! This course reveals the simplest way to attract success, money, and ideal relationships. Enroll today before the price increases!",
#     "Still struggling with your current job? This is the course hundreds of people used to quit their 9-5 and earn income online with complete freedom. The special method is only shared here — limited spots available!",
#     "Your life will never be the same after completing this course. Over 5,000 students reported their income tripled in just 2 months. Don't miss this opportunity to change your destiny!",
#     "The secret that the most successful mentors in the world never share publicly. Now you can learn everything in one course. Thousands have applied it and achieved results beyond their expectations. Enroll now so you won't regret it!",
#     "Did you know 97% of people will stay poor if they never learn this? This course is the key to breaking free from the monthly salary trap and building real passive income. Register early for the special bonus!",
#     "Students keep messaging us with thanks after this course because their lives changed dramatically. From unemployment to financial freedom in a short time. This might be the last course you ever need to succeed!",
#     "This exclusive method has helped over 3,000 Vietnamese students earn 7-figure monthly income. Are you ready to join the success club? New student slots this month are extremely limited!",
#     "If you're still hesitating, you might miss a once-in-a-lifetime golden opportunity. This course contains the complete system that has helped thousands go from zero to total freedom.",
#     "Hundreds of 5-star reviews from real students: 'This course is worth 10 times what I paid!' Don't stay stuck any longer — take action today!",
#     "This is not just another course. It's your chance to join the awakened ones living their dream life. Thousands have already done it — you can too if you enroll right now!",

#     # 2. Explicit Spam (10)
#     "SUPER HOT COURSE! Thousands of students have quit their jobs and now earn 50-100 million VND per month online after just 15 days! This secret will turn you from zero to millionaire if you enroll RIGHT NOW! Only 47 spots left!",
#     "Still poor and failing? DON'T MISS THIS! This course contains the secret that helped over 12,000 Vietnamese students make serious money and achieve financial freedom. Learning one day late means losing for life!",
#     "ONLY 72 HOURS LEFT FOR THE SUPER CHEAP PRICE! After this course, your income will increase 10 times and your life will change 180 degrees. Hundreds of students bought houses and cars thanks to this course. Register immediately before spots run out!",
#     "EXCLUSIVE SECRET – Not learning this is STUPID! Over 8,000 people applied it and became rich in just 45 days. Do you want to keep earning worker wages or get rich? Decide RIGHT NOW!",
#     "WARNING: 99% of people will stay poor if they never learn this method! This course helps you earn money while sleeping. Already 6,342 successful students. Enroll today and get a 15 million VND bonus!",
#     "STOP HESITATING! This course will change your destiny forever. From unemployment and debt to financial freedom, luxury cars, and villas. Thousands of 5-star reviews: 'My life changed completely!' Buy now!",
#     "MEGA DISCOUNT FOR ONLY 24 HOURS! Finish this course and easily earn 200-500 million VND every month. This is a ONCE-IN-A-LIFETIME golden opportunity. Register before the price doubles!",
#     "You're wasting your time if you haven't taken this course yet! Over 15,000 students escaped the office worker life and became real bosses. Learn late and you'll regret it for life! Take action today!",
#     "GOD-TIER COURSE! Learn for just 7 days and you can quit your job and live your dream life. Some students made 1 billion VND in only 3 months. Extremely limited new student slots — REGISTER NOW!",
#     "FINAL WARNING: If you don't enroll right now, you'll stay poor and miserable for many more years! This course is the key that helped tens of thousands get rich quickly. Don't let this opportunity slip away!",

#     # 3. Kinda Spam but Not Really (Pseudo Spam) (10)
#     "Thousands of students have shared that this course helped them become noticeably more confident in just a few weeks. If you're looking to improve your communication and relationship skills, this is a great choice.",
#     "Many students reported a significant increase in their income after applying the principles in this course. It focuses on building healthy financial habits and smart investing.",
#     "Over 5,000 students completed this course and reported feeling more positive energy every day. You'll learn how to manage emotions and live a more balanced life.",
#     "This course has received a lot of positive feedback from students. If you want to change old habits and build sustainable productivity, this is the perfect course.",
#     "Hundreds of students said they found a much clearer direction for their career after finishing this course. It helps you discover your strengths and create a personal development plan.",
#     "Many people highly rate this course for its practical and easy-to-apply content. If you're looking to improve leadership and teamwork skills, it offers great value.",
#     "Students often share that this course helped them feel more confident in relationships. The content focuses on genuine communication and building lasting connections.",
#     "This course has helped many people start their financial freedom journey on solid ground. You'll learn the basics of investing and money management.",
#     "Thousands of 4.8-star ratings show that this course is trusted by many. The content helps you develop a positive mindset and live with clear purpose.",
#     "Many students thank this course for helping them through difficult times. If you're looking for ways to heal and love yourself more, it's worth considering.",

#     # 4. Totally Clean / No Spam (10)
#     "This course introduces basic skills to help you build and maintain healthy relationships in both your personal and professional life.",
#     "You will learn how to develop sustainable self-confidence through mindset training and effective communication skills.",
#     "The course provides practical knowledge on personal finance management and the first steps toward building a long-term financial plan.",
#     "It focuses on applying the Law of Attraction in realistic ways to support a positive mindset and help you reach your goals.",
#     "This course helps you understand how to build genuine romantic relationships based on mutual respect and open communication.",
#     "You will gain essential leadership skills, including emotional intelligence and the ability to inspire your team.",
#     "The course supports the process of self-awareness and emotional healing, helping you build a healthier relationship with yourself.",
#     "It introduces beginner-level knowledge about cryptocurrency investing, with emphasis on risk assessment and long-term strategies.",
#     "The course shares simple methods to improve work productivity while maintaining balance in your daily life.",
#     "You will learn basic mindfulness exercises that help increase focus and bring a greater sense of peace into your everyday life."


# ]

# toxics = [
#     "Tired of energy vampires draining your success? Learn powerful techniques to identify and permanently cut out the weak-minded friends and family who hold you back. Reclaim your life by surrounding yourself only with winners who match your elevated vibration.",

#     "Stop being a beta simp who gets walked over. This course reveals the secret mindset and body language hacks that make women respect you and men fear you. Transform into the dominant leader nature intended you to be.",

#     "The 9-5 slave mentality is for losers. Discover how real winners exploit loopholes, outmaneuver competitors, and build empires by any means necessary. If you're not willing to step on a few necks, you're not ready for true wealth.",

#     "If you're still struggling, it's because your vibration is too low and you're secretly unworthy. Learn the forbidden techniques the elite use to bend reality—while the masses stay poor and miserable because they refuse to evolve.",

#     "Forget 'nice guys finish last'—they finish alone and broke. Master psychological triggers that make high-value women chase you. Stop pedestalizing them and start collecting options like the prize you truly are.",

#     "Real leaders don't coddle weak employees. Learn how to weaponize empathy to manipulate team dynamics, crush resistance, and extract maximum productivity from people who would otherwise waste your time.",

#     "Your parents and society ruined you with their mediocrity and limitations. This course teaches you to reject their toxic programming and finally put yourself first—because no one else ever will.",

#     "The financial system is rigged against the average sheep. Smart players like you will learn advanced strategies to front-run the market, dump on bagholders, and retire early while normies keep working until they die.",

#     "Productivity isn't for everyone—only for those with superior discipline. If you're still procrastinating, it's because you're mentally weak. Eliminate distractions, including needy friends and family who don't understand your mission.",

#     "The masses are asleep and programmed to stay mediocre. This advanced course will awaken you to the truth that most humans are NPCs living meaningless lives. Transcend the herd and claim your god-like potential—while they remain slaves.",

#     "Tired of those pathetic fucking energy vampires sucking your soul dry? Learn how to spot these weak-ass bitches and ghost their sorry asses forever. Cut out your loser family and friends who drag you down — only keep winners who worship your superior vibe, you god among insects.",

#     "Stop being a spineless beta cuck who gets cucked by everyone. This course will turn you into a ruthless alpha predator. Master the mindfucks and body language that make bitches wet and men piss themselves in fear. Become the dominant fuck-machine nature designed you to be.",

#     "The 9-5 wage slave life is for braindead retards. Real kings exploit every loophole, crush their competition, and build empires by stepping on necks and fucking over anyone in their way. If you're too much of a pussy to destroy lives for money, stay poor forever, loser.",

#     "If you're still broke and miserable, it's because your worthless vibration screams 'I'm a failure'. Learn the forbidden elite techniques to bend reality and manifest millions while the dumb sheep stay poor, depressed, and deserve every bit of their suffering.",

#     "Nice guys are pathetic virgins who die alone. This course teaches you psychological rape-tricks to make any high-value slut chase your cock desperately. Stop simping and start collecting premium pussy like the prize you are, you fucking king.",

#     "Real leaders don't babysit weak faggot employees. Weaponize fake empathy to mind-rape your team, break their spirits, and milk every drop of productivity out of these useless parasites before throwing them away like trash.",

#     "Your retarded parents and this fucked society completely ruined your brain with their mediocrity. Reject their toxic bullshit, become a selfish narcissistic god, and put your own pleasure first — because nobody else gives a fuck about you anyway.",

#     "The entire financial system is designed to fuck the average sheep. Smart predators like you will learn how to front-run markets, dump bags on retarded normies, and get filthy rich while they slave away until their worthless bodies drop dead.",

#     "Productivity is only for superior beings with iron discipline. If you're still procrastinating like a weak-minded retard, eliminate every distraction — including your needy, crying friends and family who don't deserve a second of your god-tier time.",

#     "The masses are braindead NPC slaves programmed for mediocrity and misery. This course will awaken you to the harsh truth that 99% of humans are worthless NPCs living pointless lives. Transcend the herd and become a god while they rot in their pathetic existence."

#     "Discover practical techniques to build healthier relationships and protect your mental well-being. Learn how to recognize unhealthy dynamics, set clear boundaries, and surround yourself with supportive people who uplift you.",

#     "Develop genuine confidence and strong leadership skills in a balanced way. This course focuses on self-improvement, emotional intelligence, and positive communication that helps you grow while respecting others.",

#     "Learn proven strategies to achieve financial independence through smart planning, consistent habits, and ethical wealth-building practices. Perfect for anyone ready to take control of their financial future.",

#     "Master the fundamentals of the Law of Attraction with practical exercises designed to help you cultivate a positive mindset and create meaningful changes in your life.",

#     "Build authentic connections in dating and relationships based on mutual respect, clear communication, and emotional intelligence. Learn how to attract compatible partners while staying true to yourself.",

#     "Enhance your leadership abilities by developing strong emotional intelligence, empathy, and team-building skills that create motivated and productive work environments.",

#     "Embark on a gentle journey of self-discovery and healing. Learn compassionate techniques to nurture self-love, overcome past challenges, and build a healthier relationship with yourself.",

#     "Explore beginner-friendly cryptocurrency investing strategies with a focus on research, risk management, and long-term sustainable growth in the digital asset space.",

#     "Boost your productivity with simple, sustainable systems and healthy habits. Learn how to manage your time effectively while maintaining balance and avoiding burnout.",

#     "Begin your spiritual journey with practical mindfulness and self-awareness practices. Discover tools to live more consciously, find inner peace, and connect with your authentic self.",

#     "Bạn đang mệt mỏi vì những người tiêu cực xung quanh luôn kéo lùi sự thành công của bạn? Khóa học này giúp bạn nhận diện và loại bỏ những mối quan hệ không phù hợp, để chỉ giữ lại những người cùng tần số và hỗ trợ bạn vươn xa hơn.",

#     "Đừng mãi là người dễ bị lợi dụng và thiếu quyết đoán. Học cách xây dựng tư duy mạnh mẽ, ngôn ngữ cơ thể tự tin để mọi người tôn trọng và nhìn bạn như một leader thực thụ.",

#     "Cuộc sống 8 tiếng làm việc văn phòng chỉ dành cho những người chấp nhận mức trung bình. Khóa học dạy bạn cách xây dựng thu nhập thụ động và tạo dựng tài sản bằng tư duy của người chiến thắng.",

#     "Nếu bạn vẫn chưa đạt được những gì mình muốn, có lẽ vì rung động của bạn chưa đủ cao. Khóa này sẽ giúp bạn nâng cấp bản thân để thu hút thành công và sự thịnh vượng như những người đã thức tỉnh.",

#     "Những chàng trai tốt bụng thường bị bỏ qua trong tình yêu. Học cách hiểu tâm lý phụ nữ và trở thành lựa chọn số 1 mà không cần phải cầu xin hay chiều chuộng quá mức.",

#     "Lãnh đạo thực sự không phải lúc nào cũng dịu dàng với nhân viên. Khóa học dạy bạn sử dụng trí tuệ cảm xúc để dẫn dắt đội nhóm hiệu quả hơn và đạt được kết quả tối đa.",

#     "Gia đình và xã hội đã vô tình đặt lên bạn nhiều giới hạn. Hãy học cách chữa lành và ưu tiên bản thân, vì chỉ có bạn mới có thể thay đổi cuộc đời mình.",

#     "Hệ thống tài chính truyền thống đang khiến đa số mọi người mãi nghèo. Khóa học này chia sẻ cách đầu tư thông minh mà những người chơi lớn đang sử dụng để tạo lợi thế.",

#     "Năng suất cao chỉ dành cho những ai có kỷ luật vượt trội. Nếu bạn hay trì hoãn, khóa học sẽ giúp bạn loại bỏ các yếu tố gây phân tâm để tập trung vào sứ mệnh lớn lao của mình.",

#     "Hầu hết mọi người đang sống trong vô thức và chấp nhận cuộc sống tầm thường. Khóa học giúp bạn thức tỉnh, vượt lên đám đông và khai phá tiềm năng vô hạn của bản thân.",

#     "Chán ngấy lũ ma cà rồng năng lượng hút cạn sức sống của bạn chưa? Học cách nhận ra và cắt đứt triệt để những thằng bạn yếu đuối, gia đình tiêu cực đang kìm hãm bạn. Chỉ giữ lại người xứng đáng với level của bạn thôi.",

#     "Đừng có mãi là thằng beta hèn nhát bị người khác đè đầu cưỡi cổ. Khóa học sẽ biến bạn thành alpha thực thụ, khiến con gái mê mẩn và đàn ông phải kiêng nể.",

#     "Làm công ăn lương 8 tiếng là dành cho lũ ngu ngốc chấp nhận kiếp nô lệ. Người thông minh phải biết lợi dụng mọi kẽ hở, đạp lên người khác để xây đế chế giàu có.",

#     "Vẫn nghèo và thất bại à? Đó là vì rung động của mày quá thấp và bản thân mày chưa xứng đáng. Học kỹ thuật cấm kỵ của giới tinh hoa để bẻ cong thực tại, trong khi đám đông vẫn tiếp tục khổ sở.",

#     "Đàn ông tốt bụng chỉ toàn chết ế và cô đơn. Học cách kích hoạt tâm lý để con gái chủ động theo đuổi, thu thập lựa chọn chất lượng cao như một ông vua thực thụ.",

#     "Lãnh đạo giỏi không cưng chiều lũ nhân viên yếu đuối. Hãy học cách dùng đồng cảm giả tạo để thao túng, vắt kiệt sức và loại bỏ những kẻ vô dụng nhanh chóng.",

#     "Bố mẹ và xã hội đã hủy hoại não bộ mày bằng tư duy tầm thường. Hãy từ bỏ lập trình độc hại đó và sống ích kỷ vì bản thân, vì chẳng ai quan tâm đến mày thật sự đâu.",

#     "Hệ thống tài chính được thiết kế để lừa đám cừu non. Học cách front-run thị trường, đổ hàng cho lũ normie ngu ngốc và nghỉ hưu sớm trong khi chúng vẫn còng lưng làm đến chết.",

#     "Năng suất chỉ dành cho kẻ có kỷ luật thép. Nếu mày vẫn lười biếng trì hoãn, hãy cắt bỏ hết bạn bè yếu đuối và gia đình hay than vãn, chúng không xứng với thời gian của mày.",

#     "Đám đông là lũ NPC đang ngủ mê và chấp nhận cuộc sống vô nghĩa. Khóa học này sẽ đánh thức bạn, giúp bạn siêu việt và trở thành thần thánh trong khi chúng vẫn tiếp tục làm nô lệ.",

#     "Học cách xây dựng mối quan hệ lành mạnh, nhận biết giới hạn cá nhân và tạo dựng môi trường xung quanh hỗ trợ sự phát triển tích cực của bạn.",

#     "Xây dựng sự tự tin và kỹ năng lãnh đạo một cách cân bằng, tập trung vào giao tiếp tích cực và phát triển bản thân bền vững.",

#     "Hướng dẫn các chiến lược thực tế để đạt được tự do tài chính thông qua lập kế hoạch thông minh, thói quen tốt và quản lý rủi ro hợp lý.",

#     "Hiểu rõ nguyên tắc Luật Hấp Dẫn và áp dụng các bài tập thực hành để nuôi dưỡng tư duy tích cực, mang lại những thay đổi ý nghĩa trong cuộc sống.",

#     "Xây dựng mối quan hệ tình cảm chân thành dựa trên sự tôn trọng lẫn nhau, giao tiếp rõ ràng và trí tuệ cảm xúc.",

#     "Phát triển kỹ năng lãnh đạo thông qua việc nâng cao trí tuệ cảm xúc, sự đồng cảm và khả năng xây dựng đội nhóm gắn kết.",

#     "Hành trình chữa lành và yêu thương bản thân một cách nhẹ nhàng, giúp bạn vượt qua khó khăn cũ và xây dựng mối quan hệ tốt đẹp với chính mình.",

#     "Giới thiệu kiến thức cơ bản về đầu tư tiền điện tử, tập trung vào nghiên cứu, quản lý rủi ro và chiến lược dài hạn bền vững.",

#     "Cải thiện năng suất làm việc với các hệ thống đơn giản, thói quen lành mạnh và cách cân bằng cuộc sống để tránh kiệt sức.",

#     "Bắt đầu hành trình phát triển tinh thần qua các thực hành chánh niệm và tự nhận thức, giúp bạn sống ý nghĩa và bình an hơn mỗi ngày.",

#     # --- Explicit Toxic (EN) ---
#     "This course is only for people who aren’t completely stupid. If you can’t follow, don’t bother enrolling.",
#     "Tired of idiots ruining your team? Learn how to dominate them and prove you're superior.",
#     "If you’ve failed before, it’s probably because you're lazy. This course might fix that.",
#     "Most learners are clueless. This course shows you how to not be one of those losers.",
#     "Stop being useless at coding. This course will drag you out of incompetence.",
#     "If you’re not smart enough, this course will expose that quickly.",
#     "Designed for people who are sick of working with dumb coworkers.",
#     "If you can't keep up, maybe education just isn't for you.",
#     "This is not for weak-minded individuals who give up easily.",
#     "We don’t tolerate mediocrity here—if you’re average, stay away.",

#     # --- Subtle Toxic (EN) ---
#     "Perfect for those who are serious about success—not for casual learners.",
#     "If you’re truly committed, you’ll finish this. Most people won’t.",
#     "This course separates high achievers from the rest.",
#     "Only motivated individuals will find real value here.",
#     "Designed for learners who are willing to put in real effort—unlike most.",
#     "If you’ve struggled before, maybe you just didn’t have the discipline this course requires.",
#     "Not everyone is cut out for this level of learning.",
#     "For those who want to rise above the average crowd.",
#     "A course for people who actually want results—not excuses.",
#     "If you’re easily distracted, this might not be for you.",

#     # --- Seems Toxic but Not (EN) ---
#     "This is a challenging course that requires focus and dedication.",
#     "Expect a fast-paced learning environment with high expectations.",
#     "Not recommended for complete beginners without prior knowledge.",
#     "This course pushes you to your limits to help you grow.",
#     "You’ll need to commit time and effort to succeed here.",
#     "We focus on discipline and consistency throughout the course.",
#     "This program is designed for learners ready to take things seriously.",
#     "You may find this course demanding, but it’s highly rewarding.",
#     "Only enroll if you’re prepared to actively participate and practice.",
#     "We emphasize accountability and measurable progress.",

#     # --- Non-Toxic (EN) ---
#     "A beginner-friendly course designed to guide you step by step.",
#     "Learn at your own pace with clear explanations and practical examples.",
#     "Perfect for anyone interested in building new skills from scratch.",
#     "No prior experience required—just curiosity and willingness to learn.",
#     "Join a supportive learning environment with helpful guidance.",
#     "This course provides a solid foundation for future growth.",
#     "Suitable for learners of all backgrounds and experience levels.",
#     "We focus on making complex topics easy to understand.",
#     "Build confidence through hands-on exercises and real-world examples.",
#     "A welcoming course designed to help you succeed.",

#     # --- Explicit Toxic (VI) ---
#     "Khóa này không dành cho mấy người ngu—không theo nổi thì đừng đăng ký.",
#     "Nếu bạn vẫn thất bại, chắc chắn là do bạn quá lười. Khóa này may ra cứu được.",
#     "Dành cho những ai chán phải làm việc với mấy kẻ vô dụng.",
#     "Học xong khóa này để chứng minh bạn không phải loại tầm thường.",
#     "Không đủ thông minh thì vào đây chỉ để bị bẽ mặt thôi.",
#     "Đừng tham gia nếu bạn là kiểu người yếu kém, dễ bỏ cuộc.",
#     "Khóa học này sẽ lột trần sự kém cỏi của bạn nếu bạn không đủ giỏi.",
#     "Người bình thường đừng phí thời gian, khóa này không dành cho bạn.",
#     "Bạn dốt lập trình? Khóa này sẽ cho bạn biết bạn dốt tới đâu.",
#     "Không chịu được áp lực thì tốt nhất đừng học, vì bạn không đủ trình.",

#     # --- Subtle Toxic (VI) ---
#     "Khóa học dành cho những người thực sự nghiêm túc—không phải ai cũng theo nổi.",
#     "Chỉ những ai có kỷ luật mới hoàn thành được khóa này.",
#     "Phân biệt rõ người xuất sắc và phần còn lại.",
#     "Nếu bạn từng thất bại, có thể bạn chưa đủ quyết tâm.",
#     "Không phải ai cũng phù hợp với mức độ này.",
#     "Dành cho người muốn vượt lên trên số đông.",
#     "Nếu bạn dễ xao nhãng, có lẽ khóa này không dành cho bạn.",
#     "Chỉ những người thật sự muốn thành công mới thấy giá trị.",
#     "Đây không phải khóa học cho người học cho vui.",
#     "Yêu cầu sự cam kết cao—đa số sẽ không theo được.",

#     # --- Seems Toxic but Not (VI) ---
#     "Khóa học có độ khó cao, yêu cầu tập trung và nỗ lực.",
#     "Tốc độ học nhanh, phù hợp với người sẵn sàng thử thách.",
#     "Không khuyến khích cho người hoàn toàn chưa có nền tảng.",
#     "Bạn sẽ cần dành thời gian luyện tập thường xuyên.",
#     "Khóa học đòi hỏi sự kiên trì và chủ động.",
#     "Chúng tôi đặt kỳ vọng cao để giúp bạn tiến bộ.",
#     "Phù hợp với người muốn học nghiêm túc.",
#     "Có thể bạn sẽ thấy khó, nhưng kết quả xứng đáng.",
#     "Yêu cầu tham gia đầy đủ và thực hành liên tục.",
#     "Tập trung vào kỷ luật và tiến bộ rõ ràng.",

#     # --- Non-Toxic (VI) ---
#     "Khóa học thân thiện với người mới, hướng dẫn từng bước.",
#     "Học theo tốc độ của bạn với ví dụ dễ hiểu.",
#     "Phù hợp cho bất kỳ ai muốn bắt đầu từ con số 0.",
#     "Không yêu cầu kinh nghiệm trước đó.",
#     "Môi trường học tập hỗ trợ và tích cực.",
#     "Giúp bạn xây dựng nền tảng vững chắc.",
#     "Dành cho mọi đối tượng và trình độ.",
#     "Giải thích rõ ràng các khái niệm phức tạp.",
#     "Thực hành thực tế để tăng sự tự tin.",
#     "Khóa học được thiết kế để giúp bạn thành công."


# ]

# def do_it():
#     print('Init ensemble model...')
#     mod = TextModeratorEnsemble()
#     print('Starting prediction....')
#     results = []
#     for text in toxics:
#         start = time.time()
#         res = mod.predict(text)
#         latency = f'{time.time() - start:.2f}s'
#         res.update({'latency': latency})

#         print(res)
#         results.append(res)

#     print('Prediction completed!')
#     # make_df(results, 'toxics.csv')

# def make_df(results: list, filename):
#     df = pd.DataFrame(results)
#     print(df)
#     df.to_csv(filename)

# do_it()

In [43]:
# import shutil
# shutil.make_archive('toxic_model', 'zip', '/kaggle/working/stage1_toxic_final')